# Hybrid Search: Combining Semantic and Lexical Retrieval

## Riverside House's next problem: finding a needle in 197 chapters

The fine-tuning arc ([`01-llm-finetuning-data-techniques.ipynb`](01-llm-finetuning-data-techniques.ipynb)
through [`03-llm-finetuning-comparison-and-decision.ipynb`](03-llm-finetuning-comparison-and-decision.ipynb))
trained an in-house editing assistant on **Riverside House**'s catalog of seven unpublished novels
(~197 chapters, ~619,000 words) — but training a model that _remembers_ the catalog is only half the problem. Editors, marketing, and new hires still
need to actually **find** the right passage: "which chapter mentions the six founding families?", "where
does Aria Voss first meet the Meridian's Promise crew?", "what does our internal style guide say about
present-tense narration?" Today they re-read chapters by hand or ask a colleague — the same gap the
fine-tuning notebook closed for _generation_, this notebook closes for **retrieval**.

That earlier notebook also established a hard constraint that still applies here: **no manuscript data
ever leaves the building**, and everything runs on hardware Riverside already owns — a laptop, not a
GPU cluster, and no hosted API in the loop. That constraint directly shapes this notebook's retrieval
stack too — every embedding call below runs a local `sentence-transformers` model in-process, never a
hosted embeddings API. The same privacy requirement that ruled out cloud fine-tuning also rules out
sending manuscript text to a third-party endpoint just to compute a vector.

Two search techniques get tried and neither is enough alone:

- **Semantic (embedding) search** finds the chapter that's _about_ a scene even if the employee's
  wording doesn't match the manuscript's — but it can bury an exact character name or invented term
  under more "generically similar" chapters.
- **Lexical (keyword) search** (BM25) nails an exact term like "Meridian's Promise" instantly — but
  misses it entirely if the employee describes the same thing in different words.

This notebook builds a **complete mental model for hybrid search in RAG** from first principles —
starting with concrete failure cases that expose the blind spots of pure semantic search and pure
keyword search, then assembling the fusion techniques that give Riverside's knowledge base the best of
both. Every concept is demonstrated on a stand-in dataset shaped like Riverside's real problem (rare
proper nouns sitting next to paraphrasable descriptions):

> **A medical knowledge base with 10 documents about symptoms, diagnoses, and treatments** — rare
> technical terms like "tachycardia" and paraphrasable concepts like "elevated blood pressure" mirror
> Riverside's rare character/place names next to paraphrasable scene descriptions.

| Step | Concept                      | Riverside's question for this section                                                            |
| ---- | ---------------------------- | ------------------------------------------------------------------------------------------------ |
| 1    | The Search Gap Problem       | If an editor searches with a rare term or a paraphrase, will search find the right chapter?      |
| 2    | Semantic/Vector Search       | Can dense embeddings find a scene when the employee doesn't use the manuscript's exact words?    |
| 3    | BM25/Lexical Search          | Can keyword search find an exact invented name no embedding model was ever trained to recognize? |
| 4    | Why Hybrid Wins              | Do we need both, or does one method already cover Riverside's real query patterns?               |
| 5    | Reciprocal Rank Fusion (RRF) | When semantic and lexical disagree on ranking, how do we merge them without one dominating?      |
| 6    | Score Normalization          | If we blend raw scores instead of ranks, how do we make them comparable at all?                  |
| 7    | Alpha Tuning                 | Is there one blend ratio for a manuscript-and-docs KB, or does it depend on the query?           |
| 8    | LangChain EnsembleRetriever  | What does Riverside's IT team actually wire up to serve this search day to day?                  |
| 9    | Advanced Patterns            | As the catalog keeps growing, what keeps search both fast and accurate?                          |
| 10   | Benchmarking                 | Before this ships to every employee, how do we know it beats "ask a colleague"?                  |

---


## Why Hybrid Retrieval?

![Hybrid retrieval storyboard showing exact identifier matching, semantic matching, and reciprocal rank fusion](images/hybrid-retrieval-storyboard.png)

Sparse retrieval is often strongest for exact identifiers and rare terms, while dense retrieval can recover semantic matches. Reciprocal rank fusion (RRF) combines their ranked lists without requiring the scores to be on the same scale.


### The Full Retrieval Topic Space — What a Complete Treatment Would Cover

Before diving in cell by cell, here's the complete landscape of "retrieval for RAG" this notebook is
choosing to build, illustrate, or explicitly skip — so any gap below is a deliberate scope choice, not
an accident:

| Sub-topic                                               | Where      | Coverage                                                              |
| ------------------------------------------------------- | ---------- | --------------------------------------------------------------------- |
| Dense/embedding (bi-encoder) retrieval + failure modes  | Parts 1–2  | Built & measured                                                      |
| Sparse/lexical retrieval (BM25, TF-IDF) + failure modes | Parts 1, 3 | Built & measured                                                      |
| Score normalization (min-max, z-score)                  | Part 6     | Built & measured                                                      |
| Fusion — weighted linear combination                    | Parts 5, 7 | Built & measured                                                      |
| Fusion — Reciprocal Rank Fusion (RRF)                   | Part 5     | Built & measured                                                      |
| Fusion — learned (a trained ranking/fusion model)       | —          | Named only — needs labeled click/relevance training data              |
| Cross-encoder reranking (2nd-stage)                     | Part 9     | Built & measured, real cross-encoder model                            |
| Two-stage retrieval (cascade pre-filter)                | Part 9     | Built & measured                                                      |
| Query expansion / rewriting                             | Part 9     | Explained, not built into a runnable pipeline                         |
| Chunking strategy's effect on retrieval                 | —          | Named only — this notebook treats each document as the retrieval unit |
| Metadata / filtered search                              | —          | Named only — no queryable metadata in this toy corpus                 |
| Vector indexing: exact vs. approximate (HNSW/IVF)       | Part 9     | Explained, not built into a runnable demo                             |
| Evaluation: Recall@K, MRR                               | Part 10    | Built & measured                                                      |
| Evaluation: nDCG (graded relevance)                     | Part 10    | Explained, not implemented — labels here are binary                   |

**Built & measured** = implemented and demonstrated with real code and measured output ·
**Explained** = illustrated but not fully built · **Named only** = explicitly out of scope, with a
one-line reason. The closing section of this notebook revisits this exact table as a full
three-tier ledger once every part has run.


## Table of Contents

1. [Part 1 — The Search Gap Problem: Where Each Method Fails](#part-1-the-search-gap-problem-where-each-method-fails)
   - [Failure Mode 1: Semantic Search Misses Exact Medical Terms](#failure-mode-1-semantic-search-misses-exact-medical-terms)
   - [Failure Mode 2: Lexical Search Misses Semantic Equivalents](#failure-mode-2-lexical-search-misses-semantic-equivalents)
   - [Common Pitfalls: Trusting One Search Method Alone](#common-pitfalls-trusting-one-search-method-alone)
2. [Part 2 — Semantic Search: Dense Vector Intuition](#part-2-semantic-search-dense-vector-intuition)
3. [Part 3 — BM25: Lexical Search with IDF Weighting](#part-3-bm25-lexical-search-with-idf-weighting)
   - [BM25 vs TF-IDF: What's the Difference?](#bm25-vs-tf-idf-whats-the-difference)
4. [Part 4 — Why Hybrid Search Wins: Complementary Strengths](#part-4-why-hybrid-search-wins-complementary-strengths)
   - [4a. Qualitative Comparison — Retrieval Method vs. Query Type](#4a-qualitative-comparison-retrieval-method-vs-query-type)
5. [Part 5 — Reciprocal Rank Fusion (RRF): The Math Behind Merging](#part-5-reciprocal-rank-fusion-rrf-the-math-behind-merging)
   - [5a. The Common Shorthand — and Why It's Incomplete](#5a-the-common-shorthand-and-why-its-incomplete)
   - [RRF vs Weighted Score Fusion](#rrf-vs-weighted-score-fusion)
   - [Common Pitfalls: Fusing Ranked Lists Naively](#common-pitfalls-fusing-ranked-lists-naively)
6. [Part 6 — Score Normalization: Making Scores Comparable](#part-6-score-normalization-making-scores-comparable)
7. [Part 7 — Alpha Tuning: Finding the Optimal Balance](#part-7-alpha-tuning-finding-the-optimal-balance)
   - [Common Pitfalls: Alpha Tuning](#common-pitfalls-alpha-tuning)
8. [Part 8 — LangChain EnsembleRetriever: Production Implementation](#part-8-langchain-ensembleretriever-production-implementation)
9. [Part 9 — Advanced Production Patterns](#part-9-advanced-production-patterns)
   - [Code Demo: Cross-Encoder Reranking — the Real Second Stage](#code-demo-cross-encoder-reranking-the-real-second-stage)
10. [Part 10 — Benchmarking: Measuring Retrieval Quality](#part-10-benchmarking-measuring-retrieval-quality)
    - [Multi-Strategy Comparison: BM25 × Dense × Hybrid](#multi-strategy-comparison-bm25-dense-hybrid-recall5-and-mrr)
11. [Summary: The Complete Hybrid Search Journey](#summary-the-complete-hybrid-search-journey)
12. [What This Notebook Covered (and What It Didn't)](#what-this-notebook-covered-and-what-it-didnt)
13. [The Decision: What Does Riverside House Actually Deploy?](#the-decision-what-does-riverside-house-actually-deploy)

> Links jump to the matching heading below. If a link doesn't scroll correctly in your Jupyter
> viewer, use `Ctrl+F` / the notebook outline panel with the section title instead.

---


In [ ]:
# Install dependencies (run once)
import subprocess, sys

required = [
    ("numpy", "numpy"),
    ("matplotlib", "matplotlib"),
    ("pandas", "pandas"),
    ("seaborn", "seaborn"),
    ("sklearn", "scikit-learn"),
    ("sentence_transformers", "sentence-transformers"),
    ("langchain", "langchain"),
    ("langchain_community", "langchain-community"),
    ("rank_bm25", "rank-bm25"),
]

for imp, pkg in required:
    try:
        __import__(imp)
        print(f"  [OK]  {pkg}")
    except ImportError:
        print(f"  Installing {pkg}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])
        print(f"  [OK]  {pkg} installed")

print("\nDependencies ready.")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer
from rank_bm25 import BM25Okapi
import warnings
import re

warnings.filterwarnings("ignore")
plt.rcParams.update({"figure.dpi": 100, "font.size": 10})
sns.set_theme(style="whitegrid", palette="muted")
np.random.seed(42)

print("Libraries loaded successfully.")

---

## Part 1 — The Search Gap Problem: Where Each Method Fails

**Riverside's question for this section:** if an editor searches with a rare term or a paraphrase,
will our knowledge-base search actually find the right chapter?

Before we jump into hybrid search, we need to understand **why** we need it. Both semantic search (dense vectors) and lexical search (keyword matching) have critical blind spots.

### Our Test Dataset: Medical Knowledge Base

We'll use 10 medical documents covering symptoms, diagnoses, and treatments. This domain is perfect for exposing search failures because:

1. **Exact medical terms matter** ("hypertension" vs "high blood pressure")
2. **Semantic understanding matters** ("chest pain" related to "cardiac arrest")
3. **Rare terms are critical** ("tachycardia" must not be diluted by common words)

Let's create our dataset and then torture-test both search methods.


In [ ]:
# Medical knowledge base — our running example
documents = [
    "Hypertension is a condition characterized by persistently elevated blood pressure readings above 140/90 mmHg.",
    "Common symptoms of diabetes include excessive thirst, frequent urination, and unexplained weight loss.",
    "Tachycardia refers to a heart rate exceeding 100 beats per minute at rest, which may indicate underlying cardiac issues.",
    "Pneumonia is an inflammatory condition of the lung affecting primarily the alveoli, often caused by bacterial or viral infection.",
    "Migraine headaches present with severe throbbing pain, often accompanied by nausea, vomiting, and sensitivity to light.",
    "Asthma is a chronic respiratory condition causing airway inflammation, leading to wheezing, coughing, and shortness of breath.",
    "Cardiac arrest occurs when the heart suddenly stops beating effectively, requiring immediate CPR and defibrillation.",
    "Type 2 diabetes develops when the body becomes resistant to insulin or doesn't produce enough insulin to maintain normal glucose levels.",
    "High blood pressure, if left untreated, can lead to serious complications including heart disease, stroke, and kidney damage.",
    "Treatment for bacterial pneumonia typically involves antibiotic therapy, rest, and adequate hydration to support recovery.",
]

print(f"Dataset: {len(documents)} medical documents\n")
for i, doc in enumerate(documents, 1):
    print(f"{i:2d}. {doc[:80]}..." if len(doc) > 80 else f"{i:2d}. {doc}")

### Failure Mode 1: Semantic Search Misses Exact Medical Terms

**Query**: "tachycardia treatment"

Semantic embeddings excel at finding **conceptually similar** content, but they can fail when:

- The query contains a **rare or technical term** ("tachycardia") that appears in only one document
- The embedding model dilutes the rare term's importance by averaging it with common words ("treatment")
- The model retrieves documents about **general cardiac issues** instead of the specific condition

Let's see this in action with a semantic search using sentence-transformers:


#### Predict first — which document will rank #1?

Before running semantic search on **"tachycardia treatment"**, predict:

- **Option A:** Document 3 (tachycardia) — exact term match
- **Option B:** Document 7 (cardiac arrest) — related cardiac concept
- **Option C:** Document 10 (pneumonia treatment) — shares "treatment"

Which will the semantic model rank highest? Commit to your guess, then run the semantic search cell to check.


In [ ]:
# ── Semantic Search Implementation ──────────────────────────────────────────────

# Load semantic embedding model
print("Loading sentence transformer model...")
semantic_model = SentenceTransformer("all-MiniLM-L6-v2")
print("[OK] Model loaded\n")

# Encode documents
print("Encoding documents...")
doc_embeddings = semantic_model.encode(documents, show_progress_bar=False)
print(f"[OK] Shape: {doc_embeddings.shape} (10 docs x 384 dims)\n")


def semantic_search(query, top_k=3):
    """Semantic search using dense embeddings"""
    query_embedding = semantic_model.encode([query], show_progress_bar=False)
    similarities = cosine_similarity(query_embedding, doc_embeddings)[0]
    ranked_indices = np.argsort(similarities)[::-1][:top_k]
    return [(idx, similarities[idx]) for idx in ranked_indices]


# Test query with rare medical term
query1 = "tachycardia treatment"
results = semantic_search(query1, top_k=5)

print(f"Query: '{query1}'\n")
print("Semantic Search Results:")
print("-" * 80)
for rank, (idx, score) in enumerate(results, 1):
    marker = "  <-- [CORRECT]" if idx == 2 else ""
    print(f"Rank {rank} (score={score:.3f}): Doc {idx+1}{marker}")
    print(f"  {documents[idx][:100]}...\n")

print("\nPrediction Check:")
print("Most people expect Doc 3 (tachycardia) to rank #1 due to exact term match.")
top_doc_idx = results[0][0]
if top_doc_idx == 2:  # Doc 3 (index 2)
    print("[OK] Semantic search correctly ranked Doc 3 first!")
else:
    print(f"[MISS] Semantic search ranked Doc {top_doc_idx+1} first instead.")
    print("   The rare term 'tachycardia' was diluted by the common word 'treatment',")
    print(
        "   causing the model to favor general cardiac concepts over the specific condition."
    )

### Failure Mode 2: Lexical Search Misses Semantic Equivalents

**Query**: "elevated blood pressure"

Keyword-based search (BM25, TF-IDF) excels at finding **exact term matches**, but fails when:

- The query uses **synonyms or paraphrases** ("elevated blood pressure" vs "hypertension")
- The document uses **medical terminology** while the query uses layman's terms
- There's **semantic equivalence without lexical overlap**

Let's implement BM25 search and expose this blind spot:


#### Predict first — will BM25 find the synonym?

Before running BM25 on **"elevated blood pressure"**, predict:

- **Option A:** Document 1 (hypertension) — semantic match, different terminology
- **Option B:** Document 9 (high blood pressure) — partial phrase match (2/3 words)

Which will BM25 rank higher? Remember, BM25 only counts **exact word matches**. Make your prediction, then run the BM25 search cell to check.


In [ ]:
# ── BM25 Lexical Search Implementation ──────────────────────────────────────────


def preprocess_text(text):
    """Simple preprocessing: lowercase + remove punctuation"""
    text = text.lower()
    text = re.sub(r"[^\w\s]", "", text)
    return text


# Tokenize documents for BM25
tokenized_docs = [preprocess_text(doc).split() for doc in documents]
bm25 = BM25Okapi(tokenized_docs)


def lexical_search(query, top_k=3):
    """BM25 keyword search"""
    tokenized_query = preprocess_text(query).split()
    scores = bm25.get_scores(tokenized_query)
    ranked_indices = np.argsort(scores)[::-1][:top_k]
    return [(idx, scores[idx]) for idx in ranked_indices]


# Test query with semantic equivalence but different terms
query2 = "elevated blood pressure"
results = lexical_search(query2, top_k=5)

print(f"Query: '{query2}'\n")
print("Lexical Search (BM25) Results:")
print("-" * 80)
for rank, (idx, score) in enumerate(results, 1):
    # Doc 0 contains the full 3-word phrase "elevated blood pressure" verbatim;
    # Doc 8 only shares "blood pressure" (2/3 query tokens) — labels reflect actual overlap.
    marker = (
        "  <-- [FULL PHRASE MATCH]"
        if idx == 0
        else "  <-- [PARTIAL MATCH]" if idx == 8 else ""
    )
    print(f"Rank {rank} (score={score:.2f}): Doc {idx+1}{marker}")
    print(f"  {documents[idx][:100]}...\n")

print("\nPrediction Check:")
print("BM25 can only match tokens verbatim — let's see which document actually wins.")
top_doc_idx = results[0][0]
if top_doc_idx == 8:  # Doc 9 (index 8)
    print(
        "[AS EXPECTED] Doc 9 (high blood pressure) ranked #1 due to exact phrase match."
    )
    print(
        "  Doc 1 (hypertension) is semantically equivalent but uses medical terminology,"
    )
    print("  so BM25 ranked it lower or missed it entirely.")
    print(
        "\nProblem: Synonym/terminology mismatch causes BM25 to miss relevant documents."
    )
elif top_doc_idx == 0:  # Doc 1 (index 0)
    print(f"[HONEST RESULT] Doc 1 ranked #1 instead of Doc 9.")
    print(
        "  Doc 1's text literally contains the phrase 'elevated blood pressure' verbatim"
    )
    print(
        "  ('...persistently elevated blood pressure readings...'), so this specific query"
    )
    print(
        "  isn't actually a clean synonym-blind-spot case for BM25 — it's a real exact-phrase"
    )
    print(
        "  match, and BM25 is right to rank it first. The true synonym gap (a layman query with"
    )
    print(
        "  zero verbatim overlap with Doc 1) still exists — see the health check after Part 1"
    )
    print("  for a cleaner isolation of that blind spot.")
else:
    print(
        f"  Doc {top_doc_idx+1} ranked #1 — an unexpected result worth investigating further."
    )

### Code Walkthrough: BM25 Lexical Search Implementation

**What just ran — 4 key patterns:**

---

**`preprocess_text(text)` — normalise vocabulary before tokenisation**
BM25 operates on bags of tokens, so "Blood," and "blood" must collapse to the same token. The function lowercases and strips punctuation with `re.sub(r"[^\w\s]", "", text)`. Without this, case differences and trailing punctuation inflate the vocabulary, causing mismatches when the same word appears in different surface forms across documents and queries.

---

**`BM25Okapi(tokenized_docs)` — pre-compute the index once at build time**
`rank_bm25.BM25Okapi` precomputes the IDF for every vocabulary term and the average document length across the corpus when the object is constructed. At query time, `bm25.get_scores(tokenized_query)` only does fast arithmetic using those cached values — no corpus scan. The `Okapi` variant adds saturation via the k₁ parameter (default 1.5) that caps the benefit of seeing the same term many times in one document, preventing very long repetitive documents from dominating the ranking.

---

**`lexical_search(query, top_k=3)` — score, sort, slice**
`get_scores()` returns one float per document. `np.argsort(scores)[::-1][:top_k]` sorts descending and slices the top k. The function returns `(doc_index, score)` pairs so callers can display scores alongside results — the score comparison against semantic cosine values in Part 4 is only possible because both functions return the same `(idx, score)` tuple format.

---

**Prediction-check print block — closed-loop pedagogy**
After each retrieval experiment in this notebook a block prints whether the outcome matched your prediction and explains the reason when it did not (e.g., "no token overlap with hypertension"). This pattern keeps the learning loop tight: a prediction you got wrong has a named cause, not a mystery.

> **Score note:** BM25 scores are unbounded — a top document may score 5.2 while cosine similarity is capped at 1.0. This scale mismatch is exactly why raw scores cannot be summed before normalisation (Part 6).


### Side-by-Side Comparison: The Search Gap

Let's visualize how semantic and lexical search produce **complementary** results on two different queries:


In [ ]:
# Compare both methods on two queries
from matplotlib.patches import Patch

test_queries = [
    ("tachycardia treatment", 2, "Rare term test"),
    ("elevated blood pressure", 0, "Synonym test"),
]

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle("The Search Gap: Semantic vs Lexical", fontsize=14, fontweight="bold")

for row, (query, target_doc, test_name) in enumerate(test_queries):
    # Semantic search
    sem_results = semantic_search(query, top_k=10)
    sem_scores = [score for _, score in sem_results]
    sem_docs = [f"Doc {idx+1}" for idx, _ in sem_results]
    sem_colors = [
        "green" if idx == target_doc else "steelblue" for idx, _ in sem_results
    ]

    # Lexical search
    lex_results = lexical_search(query, top_k=10)
    lex_scores = [score for _, score in lex_results]
    lex_docs = [f"Doc {idx+1}" for idx, _ in lex_results]
    lex_colors = ["green" if idx == target_doc else "coral" for idx, _ in lex_results]

    # Plot semantic — categorical color-coding (target vs. other) needs its own legend
    axes[row, 0].barh(sem_docs, sem_scores, color=sem_colors, alpha=0.7)
    axes[row, 0].set_xlabel("Cosine Similarity")
    axes[row, 0].set_title(f"{test_name}\nSemantic Search: '{query}'")
    axes[row, 0].invert_yaxis()
    axes[row, 0].legend(
        handles=[
            Patch(facecolor="green", alpha=0.7, label="Target document"),
            Patch(facecolor="steelblue", alpha=0.7, label="Other result"),
        ],
        loc="lower right",
        fontsize=8,
    )

    # Plot lexical — same categorical scheme, different "other" color, still needs its own legend
    axes[row, 1].barh(lex_docs, lex_scores, color=lex_colors, alpha=0.7)
    axes[row, 1].set_xlabel("BM25 Score")
    axes[row, 1].set_title(f"{test_name}\nLexical Search (BM25): '{query}'")
    axes[row, 1].invert_yaxis()
    axes[row, 1].legend(
        handles=[
            Patch(facecolor="green", alpha=0.7, label="Target document"),
            Patch(facecolor="coral", alpha=0.7, label="Other result"),
        ],
        loc="lower right",
        fontsize=8,
    )

plt.tight_layout()
plt.show()

print("\nKey Insight: Green bars show the target document.")
print("- Semantic search fails on rare terms (tachycardia)")
print("- Lexical search fails on synonyms (hypertension vs elevated blood pressure)")
print("- Neither method is universally superior — we need BOTH.")

### Common Pitfalls: Trusting One Search Method Alone

**Pitfall #1: Assuming semantic search "understands" rare technical terms**

**Bad:** Ship semantic-only search for a KB full of proper nouns, invented terms, or jargon (character names, product codes, medical terms).
**Good:** Pair semantic search with a lexical/BM25 pass whenever the corpus has rare, exact-match-critical terms.

**Why it happens:** Dense embeddings represent meaning by averaging context across many training examples. A term the model saw rarely (or never) doesn't get its own clear direction in embedding space — it gets pulled toward whatever generic topic it's closest to.

**Pitfall #2: Assuming BM25/keyword search "understands" paraphrases**

**Bad:** Ship lexical-only search and expect it to find "elevated blood pressure" when the document says "hypertension."
**Good:** Pair lexical search with semantic search whenever users might describe the same thing in different words (which is most of the time, for non-expert users).

**Why it happens:** BM25 only ever counts token overlap — it has no notion that two different strings could mean the same thing.

**Quick Health Check:** for any KB, run a query using the exact term the document contains, and a second query using a plausible paraphrase. If either method fails one of the two, you need both.


In [ ]:
# ── Quick Health Check: rare-term recall vs synonym recall ──────────────────────
# Verify the two pitfalls above are actually happening on THIS dataset, not just
# asserted in prose.

rare_term_query = "tachycardia treatment"  # exact rare term, no paraphrase
synonym_query = "elevated blood pressure"  # paraphrase, no exact term overlap

rare_term_target = 2  # Doc 3 (tachycardia)
synonym_target = 0  # Doc 1 (hypertension)

sem_rare_top1 = semantic_search(rare_term_query, top_k=1)[0][0]
lex_rare_top1 = lexical_search(rare_term_query, top_k=1)[0][0]

sem_syn_top1 = semantic_search(synonym_query, top_k=1)[0][0]
lex_syn_top1 = lexical_search(synonym_query, top_k=1)[0][0]

print("Health Check 1 — rare term ('tachycardia'):")
print(
    f"  Semantic top-1: Doc {sem_rare_top1+1}  {'[PASS]' if sem_rare_top1 == rare_term_target else '[FAIL - diluted by common word]'}"
)
print(
    f"  Lexical  top-1: Doc {lex_rare_top1+1}  {'[PASS]' if lex_rare_top1 == rare_term_target else '[FAIL]'}"
)

print(
    "\nHealth Check 2 — synonym/paraphrase ('elevated blood pressure' -> 'hypertension'):"
)
print(
    f"  Semantic top-1: Doc {sem_syn_top1+1}  {'[PASS]' if sem_syn_top1 == synonym_target else '[FAIL]'}"
)
print(
    f"  Lexical  top-1: Doc {lex_syn_top1+1}  {'[PASS]' if lex_syn_top1 == synonym_target else '[FAIL - no token overlap with hypertension]'}"
)

single_method_blind_spot = not (
    sem_rare_top1 == rare_term_target
    and lex_rare_top1 == rare_term_target
    and sem_syn_top1 == synonym_target
    and lex_syn_top1 == synonym_target
)
print(
    f"\n-> On this dataset, relying on a single method alone misses at least one target: {single_method_blind_spot}"
)
print(
    "   This is exactly why Riverside's KB needs both methods, not just the cheaper one."
)

---

## Part 2 — Semantic Search: Dense Vector Intuition

**Riverside's question for this section:** can dense embeddings find a scene when the employee
doesn't use the manuscript's exact words?

Semantic search maps text into a high-dimensional continuous space where semantically similar documents cluster together, regardless of exact word overlap.

### How It Works

1. **Embedding Model**: A neural network (e.g., BERT, sentence-transformers) encodes text into a dense vector (e.g., 384 or 768 dimensions)
2. **Similarity Metric**: Cosine similarity — $\text{similarity}(q, d) = \cos(\theta)$ — measures the angle between query and document vectors
3. **Ranking**: Documents are ranked by similarity score (range: -1 to 1, typically 0.1 to 0.9)

The key intuition: two pieces of text that mean the same thing will point in roughly the same direction in the embedding space, regardless of whether they share any words. "Car" and "automobile" produce vectors with a small angle between them because they appear in the same contexts during training. The smaller the angle, the higher the cosine similarity — so semantically equivalent texts rank near each other even with completely different vocabulary.

### Strengths and Weaknesses

**Strengths:**

- Captures semantic equivalence ("car" ≈ "automobile")
- Handles paraphrasing and synonyms naturally
- Works across languages (with multilingual models)

**Weaknesses:**

- Rare terms get diluted in high-dimensional space
- Exact keyword matches may be missed
- Requires pre-trained or fine-tuned embedding models

Let's visualize the embedding space:


In [ ]:
# ── Semantic Embedding Space Visualization ──────────────────────────────────────

from sklearn.decomposition import PCA

# Reduce embeddings to 2D for visualization
pca = PCA(n_components=2)
doc_embeddings_2d = pca.fit_transform(doc_embeddings)

# Encode a few test queries
test_queries_viz = [
    "heart problems",
    "breathing issues",
    "blood sugar",
]
query_embeddings = semantic_model.encode(test_queries_viz, show_progress_bar=False)
query_embeddings_2d = pca.transform(query_embeddings)

# Plot
plt.figure(figsize=(12, 8))
plt.scatter(
    doc_embeddings_2d[:, 0],
    doc_embeddings_2d[:, 1],
    s=100,
    alpha=0.6,
    c="steelblue",
    label="Documents",
)

# Annotate documents
for i, (x, y) in enumerate(doc_embeddings_2d):
    plt.annotate(
        f"Doc {i+1}", (x, y), xytext=(5, 5), textcoords="offset points", fontsize=9
    )

# Plot queries
plt.scatter(
    query_embeddings_2d[:, 0],
    query_embeddings_2d[:, 1],
    s=200,
    alpha=0.8,
    c="red",
    marker="*",
    label="Queries",
)

for i, (x, y) in enumerate(query_embeddings_2d):
    plt.annotate(
        test_queries_viz[i],
        (x, y),
        xytext=(5, -15),
        textcoords="offset points",
        fontsize=10,
        fontweight="bold",
        color="darkred",
    )

plt.xlabel("PCA Component 1")
plt.ylabel("PCA Component 2")
plt.title(
    "Semantic Embedding Space (384D → 2D via PCA)\nDocuments cluster by semantic similarity"
)
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print("\nNote: Documents about cardiac issues (Doc 3, 7) cluster together,")
print("as do respiratory docs (Doc 4, 6), even though they use different terms.")

---

## Part 3 — BM25: Lexical Search with IDF Weighting

**Riverside's question for this section:** can keyword search find an exact invented name (a
character, a place, an internal doc's term) no embedding model was ever trained to recognize?

BM25 scores a document by summing over the query terms, giving each term a weight based on how rare it is across the corpus and how often it appears in this specific document — with diminishing returns as frequency grows.

The full formula is: $\text{BM25}(D, Q) = \sum_{i=1}^{n} \text{IDF}(q_i) \cdot \frac{f(q_i, D) \cdot (k_1 + 1)}{f(q_i, D) + k_1 \cdot (1 - b + b \cdot |D|/\text{avgdl})}$

Three intuitions are embedded in this single equation:

**IDF gives rare terms more power.** A term that appears in only one document out of ten tells you a lot more than one that appears in all ten. The rarer the term in the corpus, the higher its score contribution — this is what makes "tachycardia" powerful as a search signal while "the" contributes almost nothing.

**Repetition has diminishing returns (TF saturation).** Mentioning "hypertension" ten times in a document is not ten times more relevant than mentioning it once. The denominator of the fraction grows sub-linearly with term frequency — after a few occurrences, additional mentions add almost nothing. This prevents documents from gaming the ranking by repeating keywords.

**Long documents don't get a free pass (length normalization).** A 1000-word document that mentions "heart rate" once is less focused on the topic than a 50-word document that does the same. The length normalization term penalizes longer documents proportionally. `b=0.75` is the standard default; `b=0` disables the penalty entirely.

### Strengths and Weaknesses

**Strengths:**

- Exact term matching (critical for technical queries)
- Rare term prioritization via IDF
- Fast and interpretable

**Weaknesses:**

- No understanding of synonyms or semantics
- Vocabulary mismatch problem
- Fails on paraphrased queries

Let's implement BM25 from scratch and compare it to sklearn's TF-IDF:


In [ ]:
# ── Manual BM25 Implementation ──────────────────────────────────────────────────


def compute_bm25_manual(query, documents, k1=1.5, b=0.75):
    """
    Manual BM25 implementation with step-by-step calculations.
    """
    # Tokenize
    tokenized_docs = [preprocess_text(doc).split() for doc in documents]
    tokenized_query = preprocess_text(query).split()

    N = len(documents)
    avgdl = np.mean([len(doc) for doc in tokenized_docs])

    # Compute IDF for each query term
    idf_scores = {}
    for term in tokenized_query:
        n_term = sum(1 for doc in tokenized_docs if term in doc)
        idf = np.log((N - n_term + 0.5) / (n_term + 0.5) + 1)
        idf_scores[term] = idf

    # Compute BM25 for each document
    scores = []
    for doc_tokens in tokenized_docs:
        doc_len = len(doc_tokens)
        score = 0.0

        for term in tokenized_query:
            if term not in idf_scores:
                continue

            # Term frequency in this document
            tf = doc_tokens.count(term)

            # BM25 formula
            numerator = tf * (k1 + 1)
            denominator = tf + k1 * (1 - b + b * (doc_len / avgdl))
            score += idf_scores[term] * (numerator / denominator)

        scores.append(score)

    return scores, idf_scores, avgdl


# Test on a query
query = "blood pressure treatment"
scores, idf_scores, avgdl = compute_bm25_manual(query, documents)

print(f"Query: '{query}'\n")
print("IDF Scores (higher = rarer term):")
for term, idf in idf_scores.items():
    print(f"  '{term}': {idf:.3f}")

print(f"\nAverage document length: {avgdl:.1f} words\n")

print("BM25 Scores:")
ranked_indices = np.argsort(scores)[::-1][:5]
for rank, idx in enumerate(ranked_indices, 1):
    print(f"Rank {rank} (score={scores[idx]:.2f}): Doc {idx+1}")
    print(f"  {documents[idx][:80]}...\n")

# Report the actual IDF ordering, not an assumed one — 'treatment' turns out to
# have a HIGHER idf here because it appears in fewer of the 10 documents than
# 'blood'/'pressure' do, even though it "feels" like a more generic word.
sorted_by_idf = sorted(idf_scores.items(), key=lambda kv: kv[1], reverse=True)
rarest_term, rarest_idf = sorted_by_idf[0]
print(
    f"Note: '{rarest_term}' actually has the HIGHEST idf here ({rarest_idf:.3f}), not the lowest —"
)
print(
    "it appears in fewer of our 10 documents than 'blood'/'pressure' do. IDF measures corpus"
)
print("rarity, not how generic a word intuitively feels.")

### BM25 vs TF-IDF: What's the Difference?

TF-IDF is simpler but lacks BM25's sophistication:

| Feature                  | TF-IDF                            | BM25                                  |
| ------------------------ | --------------------------------- | ------------------------------------- |
| Term frequency scaling   | Linear (tf × idf)                 | Saturating (diminishing returns)      |
| Document length handling | Optional normalization            | Built-in length penalty (b parameter) |
| Tuning parameters        | None                              | k₁ (saturation), b (length penalty)   |
| Typical use case         | Simple keyword search, clustering | Information retrieval, search engines |

Let's compare them side-by-side:


In [ ]:
# TF-IDF search
tfidf_vectorizer = TfidfVectorizer()
tfidf_matrix = tfidf_vectorizer.fit_transform(
    [preprocess_text(doc) for doc in documents]
)


def tfidf_search(query, top_k=5):
    query_vec = tfidf_vectorizer.transform([preprocess_text(query)])
    similarities = cosine_similarity(query_vec, tfidf_matrix)[0]
    ranked_indices = np.argsort(similarities)[::-1][:top_k]
    return [(idx, similarities[idx]) for idx in ranked_indices]


# Compare on a query
query = "diabetes insulin"

tfidf_results = tfidf_search(query, top_k=5)
bm25_results = lexical_search(query, top_k=5)

df_comparison = pd.DataFrame(
    {
        "Rank": range(1, 6),
        "TF-IDF Doc": [f"Doc {idx+1}" for idx, _ in tfidf_results],
        "TF-IDF Score": [f"{score:.3f}" for _, score in tfidf_results],
        "BM25 Doc": [f"Doc {idx+1}" for idx, _ in bm25_results],
        "BM25 Score": [f"{score:.2f}" for _, score in bm25_results],
    }
)

print(f"Query: '{query}'\n")
print(df_comparison.to_string(index=False))
print("\nBoth methods rank Doc 2 and Doc 8 highly (diabetes + insulin),")
print("but BM25's saturation and length normalization produce different scores.")

---

## Part 4 — Why Hybrid Search Wins: Complementary Strengths

**Riverside's question for this section:** do we actually need both methods, or does one already
cover Riverside's real query patterns?

Hybrid search combines semantic and lexical retrieval to create a system where:

1. **Semantic search** retrieves conceptually similar documents (even with different terminology)
2. **Lexical search** ensures exact term matches don't get lost
3. **Fusion** merges both result sets, leveraging the strengths of each

### The Coverage Theorem

For a query **Q** and document collection **D**:

- Let **S** = documents retrieved by semantic search
- Let **L** = documents retrieved by lexical search
- Let **H** = documents retrieved by hybrid search

Then: **H ⊇ (S ∪ L)** with intelligent ranking

**Key insight**: Hybrid search achieves **higher recall** than either method alone, because it captures:

- Semantically similar docs that lack exact keywords (missed by lexical)
- Exact keyword matches with low semantic similarity (missed by semantic)

### Venn Diagram: Overlap Analysis

Let's measure the overlap between semantic and lexical results across multiple queries:


### 4a. Qualitative Comparison — Retrieval Method vs. Query Type

Parts 1–3 already established two independent axes: **which retrieval method** you pick (semantic,
lexical, or hybrid) and **what kind of query** an employee actually types (a rare/exact term like
"tachycardia," or a paraphrase like "elevated blood pressure"). Before measuring anything, it's worth
writing down what each combination should look like — a hypothesis to check, not a result to trust yet:

| Retrieval method ↓ / Query type → | **Rare / exact-term query**                                                                                                                                                                    | **Paraphrase / synonym query**                                                                                                                                            |
| --------------------------------- | ---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------- | ------------------------------------------------------------------------------------------------------------------------------------------------------------------------- |
| **Semantic (dense)**              | Pros: still returns _something_ plausible. Cons: the rare term gets diluted by surrounding common words — often ranks the wrong "generically similar" document first (Part 1, Failure Mode 1). | Pros: this is its home turf — meaning survives even with zero shared vocabulary. Cons: none structural; occasional near-ties with a topically-related-but-wrong document. |
| **Lexical (BM25)**                | Pros: this is its home turf — IDF rewards the rare term directly, no training data needed for it to "know" the term. Cons: none structural, assuming the term appears verbatim somewhere.      | Pros: none structural. Cons: zero token overlap means zero score — it has no notion that two strings could mean the same thing (Part 1, Failure Mode 2).                  |
| **Hybrid (RRF)**                  | Pros: lexical's strength carries the ranking; semantic contributes nothing lost. Cons: adds a second retrieval call's latency for a query lexical alone would have solved.                     | Pros: semantic's strength carries the ranking; lexical contributes nothing lost. Cons: same latency cost, paid even when only one method was actually needed.             |

The pattern the table reveals: **hybrid has no "Cons" cell that reads "misses the query entirely"** —
every failure mode that sinks a single method's row is covered by the other method's column. The only
cost hybrid pays is running two retrieval calls instead of one, never a missed result. Further down,
the Benchmarking section's dual heatmap (Recall@5 and MRR, method × query) checks whether this
hypothesis actually held on our validation set, with real measured numbers instead of the "Pros/Cons"
guesses above.


In [ ]:
# Analyze overlap for multiple queries
test_queries_overlap = [
    "heart disease",
    "respiratory problems",
    "blood sugar levels",
    "high blood pressure",
    "lung infection",
]

overlap_stats = []

for query in test_queries_overlap:
    sem_results = set([idx for idx, _ in semantic_search(query, top_k=5)])
    lex_results = set([idx for idx, _ in lexical_search(query, top_k=5)])

    only_semantic = len(sem_results - lex_results)
    only_lexical = len(lex_results - sem_results)
    both = len(sem_results & lex_results)

    overlap_stats.append(
        {
            "Query": query,
            "Only Semantic": only_semantic,
            "Overlap": both,
            "Only Lexical": only_lexical,
        }
    )

df_overlap = pd.DataFrame(overlap_stats)

# Plot stacked bar chart
fig, ax = plt.subplots(figsize=(12, 6))
x = np.arange(len(df_overlap))
width = 0.6

p1 = ax.bar(
    x,
    df_overlap["Only Semantic"],
    width,
    label="Only Semantic",
    color="steelblue",
    alpha=0.8,
)
p2 = ax.bar(
    x,
    df_overlap["Overlap"],
    width,
    bottom=df_overlap["Only Semantic"],
    label="Overlap",
    color="purple",
    alpha=0.8,
)
p3 = ax.bar(
    x,
    df_overlap["Only Lexical"],
    width,
    bottom=df_overlap["Only Semantic"] + df_overlap["Overlap"],
    label="Only Lexical",
    color="coral",
    alpha=0.8,
)

ax.set_ylabel("Number of Documents (Top-5)")
ax.set_title("Semantic vs Lexical Retrieval Overlap\nHybrid search captures BOTH sets")
ax.set_xticks(x)
ax.set_xticklabels(df_overlap["Query"], rotation=15, ha="right")
ax.legend()
ax.set_ylim(0, 5.5)
plt.tight_layout()
plt.show()

print("\nOverlap Statistics:")
print(df_overlap.to_string(index=False))
print("\nKey Insight: Limited overlap means each method finds unique relevant docs.")
print("Hybrid search combines both, maximizing recall.")

#### What just happened — and what's the problem?

We've proven that hybrid search retrieves **more relevant documents** by combining complementary strengths:

- Semantic captures meaning but misses exact terms
- Lexical nails exact terms but ignores semantics

**The unsolved problem**: We now have **two ranked lists with incompatible score scales**:

- Semantic scores: cosine similarity ∈ [0, 1]
- BM25 scores: unbounded positive values (can be 5, 10, 50...)

How do we merge them fairly? Simply averaging won't work — a BM25 score of 10 would dominate a cosine score of 0.8, even if the semantic match is stronger.

**Next**: We need a fusion technique that handles mismatched scales. Two approaches:

1. **Reciprocal Rank Fusion (RRF)** — ignore scores, only use ranks (coming up!)
2. **Score normalization** — scale both to [0,1], then blend (Part 6)


---

## Part 5 — Reciprocal Rank Fusion (RRF): The Math Behind Merging

**Riverside's question for this section:** when semantic and lexical search disagree on ranking,
how do we merge the two lists without one method's score scale drowning out the other?

How do we merge two ranked lists with **different score scales**?

- Semantic scores: cosine similarity in [0, 1]
- BM25 scores: unbounded positive values

**Score normalization** is one approach, but **Reciprocal Rank Fusion (RRF)** is theoretically superior because it ignores raw scores entirely — it only uses ranks.

### The RRF Formula

Each document's final score is the sum of its reciprocal rank across all retrieval systems: $\text{RRF}(d) = \sum_{s \in S} \frac{1}{k + r_s(d)}$

The core intuition: rank position matters, not the raw score. A document at rank 1 in semantic and rank 2 in lexical gets a much higher combined score than one at rank 3 in both — but whether semantic gave it a cosine score of 0.9 or 0.7 is irrelevant. This makes RRF immune to scale mismatches between retrieval systems.

The constant `k` (default 60) adds a floor to the denominator, preventing the top rank from completely dominating. With a small `k`, rank 1 scores dramatically higher than rank 2 and the top result from either list dominates. With a large `k`, score differences between ranks flatten out and every position in the top 20-30 contributes meaningfully. `k=60` is empirically robust across many corpora — top ranks still matter more, but lower-ranked relevant documents aren't completely ignored.

**Why this beats simple score averaging:**

- No normalization step needed — ranks are already comparable across systems
- Resistant to outliers: one extreme BM25 score can't overwhelm the final ranking
- Proven (Cormack et al. 2009): consistently outperforms individual retrieval systems

Let's implement RRF and compare it to score-based fusion:


### 5a. The Common Shorthand — and Why It's Incomplete

RRF is frequently summarized in one line: _"it just averages the ranks from each retrieval system."_
Taken literally, that shorthand would mean: rank 1 + rank 1 → combined rank "1"; rank 3 + rank 7 →
combined rank "5" — a simple mean of two integers, where a 1-rank improvement is worth exactly as much
at rank 20 as it is at rank 1.

That's not what the formula above actually computes. Look again: $\text{RRF}(d) = \sum_{s} \frac{1}{k + r_s(d)}$
— it sums **reciprocals** of `k + rank`, not the ranks themselves, and it never divides by the number
of systems (it's a sum, not a mean). Two things fall out of that gap between the shorthand and the
real formula:

1. **The gap between rank 1 and rank 2 is worth far more than the gap between rank 20 and rank 21.**
   `1/61 − 1/62 ≈ 0.00026`, but `1/80 − 1/81 ≈ 0.00015` — the reciprocal shrinks the reward for every
   additional rank, so climbing from 2nd to 1st place matters much more than climbing from 21st to
   20th. A literal rank-average would treat every one-rank improvement identically.
2. **"Averaging" implies dividing by the number of systems — RRF never does.** Summing (not averaging)
   means a document that ranks well in _both_ systems accumulates a strictly larger score than one that
   ranks well in only one, even before either system's raw score scale enters the picture.

The shorthand isn't wrong that RRF "combines ranks instead of scores" — that part is true, and it's
exactly why RRF sidesteps the scale-mismatch problem from the previous section. It's incomplete about
_how_ those ranks get combined: through a sharply decaying reciprocal sum, not a flat average. Keep that
picture in mind for the health check further down, where a naive raw-score sum (a different mistake —
skipping ranks entirely) is measured against real RRF.


In [ ]:
# ── Reciprocal Rank Fusion Implementation ──────────────────────────────────────


def reciprocal_rank_fusion(semantic_results, lexical_results, k=60):
    """
    Merge two ranked lists using Reciprocal Rank Fusion.

    Args:
        semantic_results: List of (doc_idx, score) from semantic search
        lexical_results: List of (doc_idx, score) from lexical search
        k: Smoothing constant (default: 60)

    Returns:
        List of (doc_idx, rrf_score) sorted by RRF score
    """
    rrf_scores = {}

    # Add semantic rankings
    for rank, (doc_idx, _) in enumerate(semantic_results, start=1):
        rrf_scores[doc_idx] = rrf_scores.get(doc_idx, 0) + 1 / (k + rank)

    # Add lexical rankings
    for rank, (doc_idx, _) in enumerate(lexical_results, start=1):
        rrf_scores[doc_idx] = rrf_scores.get(doc_idx, 0) + 1 / (k + rank)

    # Sort by RRF score
    sorted_results = sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)
    return sorted_results


# Test RRF on a query
query = "heart rate problems"

sem_results = semantic_search(query, top_k=10)
lex_results = lexical_search(query, top_k=10)
rrf_results = reciprocal_rank_fusion(sem_results, lex_results, k=60)

print(f"Query: '{query}'\n")
print("Reciprocal Rank Fusion Results:")
print("-" * 80)

for rank, (doc_idx, rrf_score) in enumerate(rrf_results[:5], 1):
    # Find ranks in original lists
    sem_rank = next(
        (i + 1 for i, (idx, _) in enumerate(sem_results) if idx == doc_idx), None
    )
    lex_rank = next(
        (i + 1 for i, (idx, _) in enumerate(lex_results) if idx == doc_idx), None
    )

    print(f"Rank {rank} (RRF={rrf_score:.4f}): Doc {doc_idx+1}")
    print(f"  Semantic rank: {sem_rank if sem_rank else 'Not in top-10'}")
    print(f"  Lexical rank:  {lex_rank if lex_rank else 'Not in top-10'}")
    print(f"  {documents[doc_idx][:80]}...\n")

print("\nKey Insight: RRF boosts documents that rank well in BOTH systems.")
print("   A doc at rank (1+3) beats a doc at rank (1+10), even if the latter")
print("   has a higher semantic score. This rank-based approach is robust to")
print("   score scale mismatches and outliers — proven theoretically superior.")

### RRF vs Weighted Score Fusion

An alternative is **weighted score fusion**, which blends normalized scores using a tunable parameter α: $\text{hybrid\_score}(d) = (1 - \alpha) \cdot \text{norm(lex)} + \alpha \cdot \text{norm(sem)}$

The α parameter is the conceptual core: set it near 0 and lexical dominates; near 1 and semantic dominates. Normalization (e.g., min-max scaling to [0, 1]) is required first so BM25's unbounded values don't overwhelm cosine scores.

**Comparison:**

| Method          | Pros                                      | Cons                                   |
| --------------- | ----------------------------------------- | -------------------------------------- |
| RRF             | Rank-based, robust to outliers, no tuning | Ignores score magnitudes               |
| Weighted Fusion | Leverages score confidence, tunable α     | Requires score normalization, outliers |

Let's compare both on the same query:


In [ ]:
# ── Score Normalization & Weighted Fusion ──────────────────────────────────────


def min_max_normalize(scores):
    """Min-max normalization to [0, 1]"""
    scores = np.array(scores)
    min_score = scores.min()
    max_score = scores.max()
    if max_score == min_score:
        return np.ones_like(scores)
    return (scores - min_score) / (max_score - min_score)


def weighted_score_fusion(semantic_results, lexical_results, alpha=0.5):
    """
    Merge using normalized weighted scores.

    Args:
        semantic_results: List of (doc_idx, score)
        lexical_results: List of (doc_idx, score)
        alpha: Weight for semantic scores [0, 1]
    """
    # Extract scores and normalize
    sem_docs = [idx for idx, _ in semantic_results]
    sem_scores_raw = [score for _, score in semantic_results]
    sem_scores_norm = min_max_normalize(sem_scores_raw)

    lex_docs = [idx for idx, _ in lexical_results]
    lex_scores_raw = [score for _, score in lexical_results]
    lex_scores_norm = min_max_normalize(lex_scores_raw)

    # Build score dict
    combined_scores = {}

    for idx, norm_score in zip(sem_docs, sem_scores_norm):
        combined_scores[idx] = combined_scores.get(idx, 0) + alpha * norm_score

    for idx, norm_score in zip(lex_docs, lex_scores_norm):
        combined_scores[idx] = combined_scores.get(idx, 0) + (1 - alpha) * norm_score

    return sorted(combined_scores.items(), key=lambda x: x[1], reverse=True)


# Compare both methods
query = "tachycardia treatment"

sem_results = semantic_search(query, top_k=10)
lex_results = lexical_search(query, top_k=10)

rrf_results = reciprocal_rank_fusion(sem_results, lex_results, k=60)
weighted_results = weighted_score_fusion(sem_results, lex_results, alpha=0.5)

# Display side-by-side
df_compare = pd.DataFrame(
    {
        "Rank": range(1, 6),
        "RRF Doc": [f"Doc {idx+1}" for idx, _ in rrf_results[:5]],
        "RRF Score": [f"{score:.4f}" for _, score in rrf_results[:5]],
        "Weighted Doc": [f"Doc {idx+1}" for idx, _ in weighted_results[:5]],
        "Weighted Score": [f"{score:.4f}" for _, score in weighted_results[:5]],
    }
)

print(f"Query: '{query}'\n")
print("RRF vs Weighted Score Fusion (α=0.5):\n")
print(df_compare.to_string(index=False))
print("\nBoth methods often produce similar top results, but RRF is more robust")
print("   to outliers and doesn't require score normalization tuning.")

### Proof: RRF vs Weighted Fusion on This Dataset

Let's measure which fusion method actually performs better on our validation set using **Recall@5** as the metric.


In [ ]:
# ── Validation: RRF vs Weighted Fusion ──────────────────────────────────────────

# Validation set for measuring retrieval quality
validation_queries_rrf = [
    ("heart rate over 100 bpm", [2]),  # tachycardia
    ("high blood pressure", [0, 8]),  # hypertension + high blood pressure
    ("diabetes insulin problems", [1, 7]),  # diabetes symptoms + type 2
    ("lung infection", [3, 9]),  # pneumonia + treatment
    ("severe headache nausea", [4]),  # migraine
]


def compute_recall_at_k(results, relevant_docs, k=5):
    """Compute recall@K: fraction of relevant docs in top K"""
    retrieved = set([idx for idx, _ in results[:k]])
    relevant = set(relevant_docs)
    if len(relevant) == 0:
        return 0.0
    return len(retrieved & relevant) / len(relevant)


# Measure RRF performance
rrf_recalls = []
for query, relevant_docs in validation_queries_rrf:
    sem_results = semantic_search(query, top_k=10)
    lex_results = lexical_search(query, top_k=10)
    rrf_results = reciprocal_rank_fusion(sem_results, lex_results, k=60)
    recall = compute_recall_at_k(rrf_results, relevant_docs, k=5)
    rrf_recalls.append(recall)

avg_rrf_recall = np.mean(rrf_recalls)

# Measure Weighted Fusion performance (α=0.5)
weighted_recalls = []
for query, relevant_docs in validation_queries_rrf:
    sem_results = semantic_search(query, top_k=10)
    lex_results = lexical_search(query, top_k=10)
    weighted_results = weighted_score_fusion(sem_results, lex_results, alpha=0.5)
    recall = compute_recall_at_k(weighted_results, relevant_docs, k=5)
    weighted_recalls.append(recall)

avg_weighted_recall = np.mean(weighted_recalls)

# Display results
print("Fusion Method Comparison on Validation Set")
print("=" * 80)
print(f"RRF (k=60):                 Recall@5 = {avg_rrf_recall:.3f}")
print(f"Weighted Fusion (α=0.5):    Recall@5 = {avg_weighted_recall:.3f}")
print(f"Delta (RRF advantage):      {(avg_rrf_recall - avg_weighted_recall):.3f}")
print("\nKey Insight: RRF's rank-only approach is robust to score-scale mismatches.")
print("   It typically matches or outperforms weighted fusion without requiring")
print("   normalization tuning. Weighted fusion can beat RRF if α is optimally tuned.")
print(
    f"\n   On this dataset: {'RRF wins' if avg_rrf_recall >= avg_weighted_recall else 'Weighted wins'} by {abs(avg_rrf_recall - avg_weighted_recall):.1%}"
)

In [ ]:
# Your turn — RRF constant k
# CHANGE k_val (try 10, 60, 200) and observe how rank weighting shifts.
#    Smaller k = steep decay (top ranks dominate); larger k = gentle decay.

k_val = 60  # ← try 10 (steep) or 200 (gentle)

query_ex = "heart rate problems"
sem_ex = semantic_search(query_ex, top_k=10)
lex_ex = lexical_search(query_ex, top_k=10)
rrf_ex = reciprocal_rank_fusion(sem_ex, lex_ex, k=k_val)

print(f"RRF with k={k_val}  (query: '{query_ex}'):")
for rank, (doc_idx, score) in enumerate(rrf_ex[:5], 1):
    print(f"  Rank {rank}: Doc {doc_idx+1}  (RRF score={score:.4f})")

print(
    f"\nKey insight: k={k_val} {'favours top ranks strongly (steep decay)' if k_val < 30 else 'distributes weight more evenly (gentle decay)' if k_val > 100 else 'balances rank weight well (default)'}"
)
print("k=60 is empirically robust across many retrieval corpora.")

### Common Pitfalls: Fusing Ranked Lists Naively

**Pitfall #1: Summing raw scores without normalizing first**

**Bad:** `combined_score = bm25_score + cosine_score` — BM25 is unbounded (can be 5, 10, 50+) while cosine similarity lives in [0, 1], so BM25 silently dominates every ranking.
**Good:** Either normalize both score sets to a comparable range before blending (Part 6), or skip scores entirely and fuse by rank (RRF).

**Pitfall #2: Assuming RRF and weighted fusion always agree**

**Bad:** Pick RRF (or weighted fusion) once and never re-check against the other on your own validation queries.
**Good:** Measure both on a labeled set for your domain — the "theoretically superior" method doesn't always win in practice on a specific dataset (we measured this above; check which one won on ours).

**Quick Health Check:** actually compute a naive raw-score sum (no normalization) and confirm it produces a different, worse ranking than RRF — don't take "BM25 will dominate" on faith.


In [ ]:
# ── Quick Health Check: naive raw-score averaging vs RRF ────────────────────────
# Prove Pitfall #1 rather than asserting it: fuse with UN-normalized raw scores
# (no min-max, no z-score) and compare recall against RRF on the same validation set.


def naive_raw_fusion(semantic_results, lexical_results):
    """Sum RAW scores with no normalization — the mistake described above."""
    combined = {}
    for idx, score in semantic_results:
        combined[idx] = combined.get(idx, 0) + score
    for idx, score in lexical_results:
        combined[idx] = combined.get(idx, 0) + score
    return sorted(combined.items(), key=lambda x: x[1], reverse=True)


naive_recalls = []
for query, relevant_docs in validation_queries_rrf:
    sem_results = semantic_search(query, top_k=10)
    lex_results = lexical_search(query, top_k=10)
    naive_results = naive_raw_fusion(sem_results, lex_results)
    naive_recalls.append(compute_recall_at_k(naive_results, relevant_docs, k=5))

avg_naive_recall = np.mean(naive_recalls)

print("Naive raw-score fusion (no normalization) vs RRF vs Weighted Fusion")
print("=" * 80)
print(f"Naive raw sum (BUG):        Recall@5 = {avg_naive_recall:.3f}")
print(f"RRF (k=60):                 Recall@5 = {avg_rrf_recall:.3f}")
print(f"Weighted Fusion (α=0.5):    Recall@5 = {avg_weighted_recall:.3f}")

if avg_naive_recall < min(avg_rrf_recall, avg_weighted_recall):
    print("\n-> Confirmed: un-normalized raw-score fusion measurably underperforms")
    print("   both RRF and normalized weighted fusion on this dataset — BM25's")
    print(
        "   unbounded scores really do drown out cosine similarity when summed directly."
    )
elif avg_naive_recall == min(avg_rrf_recall, avg_weighted_recall):
    print("\n-> On this small validation set, naive fusion tied the weaker of the two")
    print("   proper methods — still never normalize raw scores in production, this")
    print("   dataset is just too small to always expose the gap.")
else:
    print(
        "\n-> Surprising on this tiny validation set: naive fusion didn't underperform"
    )
    print(
        "   here, but that's a property of this specific 5-query sample, not a reason"
    )
    print("   to skip normalization — BM25's unbounded scale is still a latent bug.")

---

## Part 6 — Score Normalization: Making Scores Comparable

**Riverside's question for this section:** if we do want to blend raw scores instead of ranks, how
do we make cosine similarity and BM25 scores comparable in the first place?

When using weighted score fusion, scores from different systems must be scaled to the same range before blending. Two common techniques:

**Min-Max Normalization** maps every score to [0, 1] by measuring its position between the observed min and max: $\text{norm}(x) = \frac{x - \min(X)}{\max(X) - \min(X)}$

The intuition: the best-performing document in the result set scores 1.0, the worst scores 0.0, and everything else falls proportionally in between. The weakness is sensitivity to outliers — one extremely high BM25 score stretches the denominator and compresses all other scores toward 0.

**Z-Score Normalization** expresses each score as a distance from the mean in standard deviation units: $z(x) = \frac{x - \mu}{\sigma}$

The intuition: outliers produce large z-scores but don't compress the rest of the distribution. The tradeoff is that z-scores can be negative, which requires extra handling (e.g., shifting by the minimum) before using them as blend weights.

For hybrid search, min-max is more common because its [0, 1] output maps cleanly onto α-weighted blending. RRF sidesteps this entire problem by discarding scores altogether and working only with ranks.

Let's visualize how normalization affects score distributions:


In [ ]:
# ── Score Normalization Visualization ──────────────────────────────────────────


def z_score_normalize(scores):
    """Z-score normalization (mean=0, std=1)"""
    scores = np.array(scores)
    mean = scores.mean()
    std = scores.std()
    if std == 0:
        return np.zeros_like(scores)
    return (scores - mean) / std


# Get scores from a query
query = "diabetes symptoms"
sem_results = semantic_search(query, top_k=10)
lex_results = lexical_search(query, top_k=10)

sem_scores_raw = [score for _, score in sem_results]
lex_scores_raw = [score for _, score in lex_results]

# Normalize
sem_scores_minmax = min_max_normalize(sem_scores_raw)
sem_scores_zscore = z_score_normalize(sem_scores_raw)

lex_scores_minmax = min_max_normalize(lex_scores_raw)
lex_scores_zscore = z_score_normalize(lex_scores_raw)

# Plot
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
fig.suptitle(
    f"Score Normalization Comparison\nQuery: '{query}'", fontsize=14, fontweight="bold"
)

# Row 1: Semantic
axes[0, 0].bar(range(10), sem_scores_raw, color="steelblue", alpha=0.7)
axes[0, 0].set_title("Semantic Scores (Raw)")
axes[0, 0].set_ylabel("Cosine Similarity")
axes[0, 0].set_ylim([0, 1])

axes[0, 1].bar(range(10), sem_scores_minmax, color="green", alpha=0.7)
axes[0, 1].set_title("Semantic Scores (Min-Max)")
axes[0, 1].set_ylim([0, 1])

axes[0, 2].bar(range(10), sem_scores_zscore, color="purple", alpha=0.7)
axes[0, 2].set_title("Semantic Scores (Z-Score)")
axes[0, 2].axhline(0, color="red", linestyle="--", linewidth=1)

# Row 2: Lexical
axes[1, 0].bar(range(10), lex_scores_raw, color="coral", alpha=0.7)
axes[1, 0].set_title("Lexical Scores (Raw BM25)")
axes[1, 0].set_ylabel("BM25 Score")
axes[1, 0].set_xlabel("Document Rank")

axes[1, 1].bar(range(10), lex_scores_minmax, color="green", alpha=0.7)
axes[1, 1].set_title("Lexical Scores (Min-Max)")
axes[1, 1].set_xlabel("Document Rank")
axes[1, 1].set_ylim([0, 1])

axes[1, 2].bar(range(10), lex_scores_zscore, color="purple", alpha=0.7)
axes[1, 2].set_title("Lexical Scores (Z-Score)")
axes[1, 2].set_xlabel("Document Rank")
axes[1, 2].axhline(0, color="red", linestyle="--", linewidth=1)

plt.tight_layout()
plt.show()

print("\nKey Insights:")
print("   Min-Max normalization:")
print("     [+] Maps scores to [0,1], preserving relative distances")
print("     [-] Sensitive to outliers (one huge score compresses the rest)")
print("\n   Z-Score normalization:")
print("     [+] Centers at 0 with std=1, robust to outliers")
print("     [-] Can produce negative scores (need shifting for weighted fusion)")
print("\n   Raw semantic scores already in [0,1], but BM25 is unbounded.")
print("   For weighted fusion, min-max is more common; RRF avoids this entirely.")

---

## Part 7 — Alpha Tuning: Finding the Optimal Balance

**Riverside's question for this section:** is there one blend ratio that works best for a
manuscript-and-internal-docs knowledge base, or does it depend on the kind of question being asked?

The parameter **α** controls the semantic/lexical blend: $\text{hybrid\_score} = (1 - \alpha) \cdot \text{lexical} + \alpha \cdot \text{semantic}$

The intuition is direct: α=0 is pure BM25, α=1 is pure semantic embeddings, and values in between weight both. The right α depends on your content and query patterns — domains where exact terminology matters (medical, legal, code) tend toward lower α; conversational or synonym-rich queries tend toward higher α.

**How to choose α:**

1. **Domain knowledge**: Technical docs → lower α (favor lexical), conversational → higher α
2. **Validation set**: Test multiple α values on labeled query-document pairs
3. **A/B testing**: Measure user satisfaction (clicks, dwell time) for different α

### Validation Experiment: Alpha Sweep

We'll create a small validation set with labeled relevance judgments, then sweep α from 0 to 1 and measure **Recall@5** (fraction of relevant docs retrieved in top-5).


#### So we can normalize scores — but what's the right α?

We've seen how to make scores comparable via normalization. But now we face a new question:

**What blend ratio should we use?**

- α = 0 → pure lexical (ignore semantic)
- α = 0.5 → equal weight (50/50)
- α = 1 → pure semantic (ignore lexical)

The optimal α depends on your **domain** and **query patterns**:

- Technical docs (code, medical) → favor lexical (lower α)
- Conversational queries → favor semantic (higher α)

**Next**: We'll run an empirical validation experiment to find the optimal α for our medical dataset by sweeping from 0 to 1 and measuring Recall@5.


In [ ]:
# ── Alpha Tuning Experiment ──────────────────────────────────────────────────

# Validation set: (query, relevant_doc_indices)
validation_queries = [
    ("heart rate over 100 bpm", [2]),  # tachycardia
    ("high blood pressure", [0, 8]),  # hypertension + high blood pressure
    ("diabetes insulin problems", [1, 7]),  # diabetes symptoms + type 2
    ("lung infection", [3, 9]),  # pneumonia + treatment
    ("severe headache nausea", [4]),  # migraine
]

# Sweep alpha from 0 to 1
alpha_values = np.linspace(0, 1, 21)  # 0.0, 0.05, 0.1, ..., 1.0
recall_scores = []

for alpha in alpha_values:
    recalls = []
    for query, relevant_docs in validation_queries:
        sem_results = semantic_search(query, top_k=10)
        lex_results = lexical_search(query, top_k=10)
        hybrid_results = weighted_score_fusion(sem_results, lex_results, alpha=alpha)
        recall = compute_recall_at_k(hybrid_results, relevant_docs, k=5)
        recalls.append(recall)

    avg_recall = np.mean(recalls)
    recall_scores.append(avg_recall)

# Find optimal alpha
optimal_alpha = alpha_values[np.argmax(recall_scores)]
max_recall = max(recall_scores)

# Plot
plt.figure(figsize=(12, 6))
plt.plot(
    alpha_values, recall_scores, "o-", linewidth=2, markersize=6, color="steelblue"
)
plt.axvline(
    optimal_alpha,
    color="red",
    linestyle="--",
    linewidth=2,
    label=f"Optimal α={optimal_alpha:.2f} (Recall@5={max_recall:.2f})",
)
plt.xlabel("Alpha (Semantic Weight)")
plt.ylabel("Average Recall@5")
plt.title("Alpha Tuning Curve\nFinding the optimal semantic/lexical balance")
plt.xticks(alpha_values[::2], [f"{a:.1f}" for a in alpha_values[::2]])
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

print(f"\nOptimal α found: {optimal_alpha:.2f}")
print(f"   Maximum Recall@5: {max_recall:.2f}")
print(f"\nBaseline comparisons:")
print(f"  α=0.0 (pure lexical):  Recall@5={recall_scores[0]:.2f}")
print(f"  α=0.5 (balanced):      Recall@5={recall_scores[10]:.2f}")
print(f"  α=1.0 (pure semantic): Recall@5={recall_scores[-1]:.2f}")
print(f"\nInterpretation:")
recall_spread = max(recall_scores) - min(recall_scores)
if recall_spread < 0.05:
    print(
        f"  [HONEST RESULT] Recall@5 is flat ({min(recall_scores):.2f}-{max(recall_scores):.2f}) across"
    )
    print(
        "  EVERY alpha from 0.0 to 1.0 — this validation set doesn't actually discriminate between"
    )
    print(
        "  blend ratios. With only 10 documents and 5 validation queries with 1-2 relevant docs"
    )
    print(
        "  each, top-5 retrieval is wide enough that nearly any reasonable ranking finds them all."
    )
    print(
        "  The 'optimal α=0.00' reported above is an artifact of argmax tie-breaking to the first"
    )
    print(
        "  index, NOT evidence that lexical search is actually better here. A real alpha-tuning"
    )
    print(
        "  decision needs either a larger corpus, a stricter K (Recall@1 or Recall@2), or harder"
    )
    print("  validation queries with more distractors than this toy set has.")
elif optimal_alpha < 0.3:
    print(
        f"  Lexical-dominant (α={optimal_alpha:.2f}): Medical queries favor exact term matching"
    )
elif optimal_alpha > 0.7:
    print(
        f"  Semantic-dominant (α={optimal_alpha:.2f}): Queries benefit from meaning over exact words"
    )
else:
    print(
        f"  Balanced (α={optimal_alpha:.2f}): Best results from combining both methods equally"
    )
print("\n  For production: Tune α on 50-100 labeled query-doc pairs from your domain.")

### Your turn — manually tune α

The sweep above found the _optimal_ α for our validation set. Now feel the tradeoff directly: change `alpha_manual` below and observe how the top-5 results shift between exact-term matches and semantic synonyms for the query `"elevated blood pressure"`.

**Predict before running:** at α=0.0 (pure lexical), which document will rank #1 — Doc 1 (uses "hypertension") or Doc 9 (uses "high blood pressure")? Does your answer change at α=1.0?


In [ ]:
# Your turn — blend ratio alpha
# CHANGE alpha_manual (0.0 = pure lexical, 1.0 = pure semantic) and observe
#    how results shift between exact-term and synonym matching.

alpha_manual = 0.5  # ← try 0.0, 0.3, 0.7, 1.0

query_tune = "elevated blood pressure"
sem_tune = semantic_search(query_tune, top_k=10)
lex_tune = lexical_search(query_tune, top_k=10)

# Normalise scores before blending
sem_norm = min_max_normalize([s for _, s in sem_tune])
lex_norm = min_max_normalize([s for _, s in lex_tune])
sem_scored = [(idx, alpha_manual * s) for (idx, _), s in zip(sem_tune, sem_norm)]
lex_scored = [(idx, (1 - alpha_manual) * s) for (idx, _), s in zip(lex_tune, lex_norm)]

all_scores = {}
for idx, s in sem_scored + lex_scored:
    all_scores[idx] = all_scores.get(idx, 0) + s
blended = sorted(all_scores.items(), key=lambda x: x[1], reverse=True)[:5]

print(f"α={alpha_manual:.1f} blend (query: '{query_tune}'):")
print(
    f"  {int((1-alpha_manual)*100)}% lexical weight + {int(alpha_manual*100)}% semantic weight"
)
for rank, (doc_idx, score) in enumerate(blended, 1):
    doc_snippet = documents[doc_idx][:60] + "..."
    print(f"  Rank {rank}: Doc {doc_idx+1}  score={score:.3f}  '{doc_snippet}'")
print(
    f"\n→ α=0.0 favours 'blood pressure' exact match; α=1.0 favours 'hypertension' synonym"
)

### Common Pitfalls: Alpha Tuning

**Pitfall #1: Hardcoding α=0.5 "to be safe"**

**Bad:** Pick a round-number α without ever validating it against labeled queries from your actual domain.
**Good:** Sweep α against a validation set that reflects your real query mix, the way we did above — the optimal value is rarely exactly 0.5.

**Pitfall #2: Assuming the optimal α from one validation set generalizes forever**

**Bad:** Tune α once on 5 launch-day queries and never revisit it as the corpus and query patterns grow (Riverside will keep publishing new manuscripts and adding internal docs).
**Good:** Re-validate α periodically on a held-out sample, and treat the tuned value as a hypothesis to keep checking, not a constant to freeze.

**Quick Health Check:** test the "optimal" α found above against a small held-out set of _different_ queries and report whether it still wins — honestly, including if it doesn't.


In [ ]:
# ── Quick Health Check: does the tuned alpha generalize to new queries? ─────────
# The sweep above optimized alpha on `validation_queries`. Test that same alpha on
# a DIFFERENT, held-out set of queries and report the real result — including if
# it doesn't generalize as well as the sweep implied.

held_out_queries = [
    ("chest discomfort and rapid heartbeat", [6]),  # cardiac arrest
    ("wheezing and shortness of breath", [5]),  # asthma
    ("frequent urination and thirst", [1]),  # diabetes symptoms
]


def recall_at_alpha(alpha, queries):
    recalls = []
    for query, relevant_docs in queries:
        sem_results = semantic_search(query, top_k=10)
        lex_results = lexical_search(query, top_k=10)
        hybrid_results = weighted_score_fusion(sem_results, lex_results, alpha=alpha)
        recalls.append(compute_recall_at_k(hybrid_results, relevant_docs, k=5))
    return np.mean(recalls)


held_out_at_optimal = recall_at_alpha(optimal_alpha, held_out_queries)
held_out_at_default = recall_at_alpha(0.5, held_out_queries)

print(
    f"Tuned α={optimal_alpha:.2f} (from the 5-query sweep) on 3 NEW held-out queries:"
)
print(f"  Recall@5 at tuned α={optimal_alpha:.2f}:  {held_out_at_optimal:.3f}")
print(f"  Recall@5 at default α=0.50:  {held_out_at_default:.3f}")

if held_out_at_optimal >= held_out_at_default:
    print("\n-> The tuned α still holds up on held-out queries — but 3 queries is a")
    print("   tiny sample. Riverside should keep re-validating as real usage grows.")
else:
    print("\n-> Honest result: the α tuned on 5 launch queries did NOT generalize")
    print("   better than the default 0.5 on these 3 held-out queries. With only")
    print("   5 validation queries, the 'optimal' α is likely overfit to that small")
    print(
        "   sample — Riverside needs a bigger labeled set before trusting one number."
    )

---

## Part 8 — LangChain EnsembleRetriever: Production Implementation

**Riverside's question for this section:** what does Riverside's IT team actually wire up to serve
search across 197 chapters plus internal docs, day to day?

LangChain provides `EnsembleRetriever` which combines multiple retrievers with configurable weights. Under the hood, it:

1. Runs each retriever independently
2. Normalizes scores to [0, 1] using min-max
3. Computes weighted average: `(1-α) * lexical + α * semantic`
4. Merges and re-ranks results

Let's build a complete RAG pipeline with hybrid search:


In [ ]:
# ── EnsembleRetriever: Production Hybrid Search Pipeline ─────────────────────

from langchain_core.documents import Document
from langchain_community.retrievers.bm25 import BM25Retriever
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings


class EnsembleRetriever:
    """Minimal Reciprocal Rank Fusion ensemble retriever (no langchain.retrievers dep)."""

    def __init__(self, retrievers, weights=None):
        self.retrievers = retrievers
        self.weights = weights or [1.0 / len(retrievers)] * len(retrievers)

    def invoke(self, query: str, k: int = 5):
        rrf: dict = {}
        for retriever, weight in zip(self.retrievers, self.weights):
            docs = retriever.invoke(query)
            for rank, doc in enumerate(docs, 1):
                key = doc.page_content
                rrf[key] = rrf.get(key, {"score": 0.0, "doc": doc})
                rrf[key]["score"] += weight * (1.0 / (60 + rank))
        ranked = sorted(rrf.values(), key=lambda x: x["score"], reverse=True)
        return [item["doc"] for item in ranked[:k]]


# Convert documents to LangChain format
langchain_docs = [
    Document(page_content=doc, metadata={"doc_id": i})
    for i, doc in enumerate(documents)
]

# Initialize embeddings
print("Initializing embeddings...")
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
print("[OK]\n")

# Create vector store
print("Building FAISS vector store...")
vectorstore = FAISS.from_documents(langchain_docs, embeddings)
semantic_retriever = vectorstore.as_retriever(search_kwargs={"k": 5})
print("[OK]\n")

# Create BM25 retriever
print("Building BM25 retriever...")
bm25_retriever = BM25Retriever.from_documents(langchain_docs)
bm25_retriever.k = 5
print("[OK]\n")

# Create ensemble retriever with custom weights
# weights = [lexical_weight, semantic_weight]
print("Creating EnsembleRetriever (weights=[0.3, 0.7])...")
ensemble_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, semantic_retriever],
    weights=[0.3, 0.7],  # 30% lexical, 70% semantic
)
print("[OK]\n")

# Test queries on the medical demo dataset
test_queries_ensemble = [
    "tachycardia treatment",
    "elevated blood pressure",
    "diabetes symptoms",
]

for query in test_queries_ensemble:
    print(f"Query: '{query}'")
    print("-" * 80)

    results = ensemble_retriever.invoke(query)

    for rank, doc in enumerate(results[:3], 1):
        doc_id = doc.metadata["doc_id"]
        print(f"Rank {rank}: Doc {doc_id+1}")
        print(f"  {doc.page_content[:100]}...\n")

    print()

print("\nEnsembleRetriever successfully combines semantic and lexical search!")
print("   Adjust weights=[lex, sem] to tune for your domain.")
print("   For production: wrap with a chain for LLM-powered RAG.")
print()

# ── Riverside analogy queries ────────────────────────────────────────────────
# The medical KB is a stand-in.  The same EnsembleRetriever (different docs) would serve
# these actual Riverside queries — shown here so the mapping from toy to real is explicit.
RIVERSIDE_QUERY_ANALOGY = [
    # (novel,           rare-term / lexical query,                   paraphrase / semantic query)
    ("Sci-fi", "Meridian's Promise", "the generation ship on the long voyage"),
    ("Fantasy", "Kerra Valmont", "the unbound resonant who can touch all five tides"),
    ("Mystery", "six founding families", "original settlers who built Ashmont Bay"),
    ("Historical", "Wei Lian", "silk merchant's daughter traveling the Silk Road"),
    ("Cyberpunk", "Project Drift", "corporate program to steal human consciousness"),
    ("Horror", "Eleanor Vance", "the keeper who made a bargain with the hollow"),
    ("Literary", "Observer", "the deep-dwelling intelligence beneath Willowport"),
]
print(
    "\nRiverside query analogy — how the same retriever applies to the actual catalog:"
)
print(
    f"{'Novel':<12}  {'Rare-term query (lexical wins)':<40}  {'Paraphrase query (semantic wins)'}"
)
print("-" * 100)
for novel, rare, paraphrase in RIVERSIDE_QUERY_ANALOGY:
    print(f"{novel:<12}  {rare:<40}  {paraphrase}")

### Code Walkthrough: EnsembleRetriever — Production Hybrid Search Pipeline

**What just ran — 5 key patterns:**

---

**`EnsembleRetriever.invoke(query, k)` — weighted RRF across retriever list**
The method iterates over every sub-retriever, collects its ranked document list, and accumulates `weight × 1/(60 + rank)` into a dict keyed by `page_content`. By multiplying by `weight` before accumulating, a retriever with `weight=0.7` contributes more than one with `weight=0.3`. Documents that rank high in multiple retrievers accumulate the largest combined scores — weighted RRF in three lines of Python.

---

**`HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")` — same encoder, LangChain interface**
This is the identical sentence-transformer model used throughout the notebook, wrapped in LangChain's `Embeddings` protocol so `FAISS.from_documents` can call it. The key invariant: the corpus and live queries share the same embedding function, so their vectors live in the same space.

---

**`FAISS.from_documents(langchain_docs, embeddings)` — one-call vector index**
FAISS builds a flat exact-search index over the 10 encoded document vectors. For a production corpus (100K+ documents) you would swap to an approximate index (`IndexIVFFlat`, `HNSW`) or a managed vector store (Pinecone, Weaviate), but `as_retriever()` at query time is identical — the production upgrade is purely in the index structure, not the calling code.

---

**`BM25Retriever.from_documents(langchain_docs)` — BM25 behind LangChain protocol**
This wraps `rank_bm25.BM25Okapi` behind LangChain's `BaseRetriever` interface. Setting `.k = 5` caps the candidate list before fusion. In production raise this to 50–200 to give RRF a wider candidate pool across which it can identify documents that rank well in both systems simultaneously.

---

**Riverside analogy table — toy corpus to real catalog**
The final block maps each medical query to an analogous Riverside manuscript query. The algorithm does not change whether the corpus is 10 medical documents or 197 chapters — only the domain differs. This is the "so what" bridge: the same `EnsembleRetriever(retrievers=[bm25_retriever, semantic_retriever])` call serves both.

> **Shape/API note:** `EnsembleRetriever.invoke` returns `Document` objects, not `(index, score)` tuples. Use `.page_content` for text and `.metadata["doc_id"]` for the original document index.


---

## Part 9 — Advanced Production Patterns

**Riverside's question for this section:** as the catalog grows past 197 chapters (new manuscripts,
revisions, internal docs), what keeps search both fast and accurate?

Beyond basic hybrid search, production RAG systems use several advanced techniques. This notebook
builds a real, runnable demo for the ones marked **(built below)**; the rest get an accurate
explanation and a named reason they stop there — see the topic-space table near the top of this
notebook and the closing coverage ledger for the full breakdown.

### 1. Two-Stage Retrieval (Cascade) — _(built below)_

**Problem**: Running semantic search on millions of documents is slow (embedding inference + vector search).

**Solution**:

1. **Stage 1**: Fast lexical pre-filter (BM25) retrieves top-100 candidates
2. **Stage 2**: Semantic reranking on the 100 candidates

**Benefits**: 10-100x faster than full semantic search, 95% of quality retained

### 2. Query Expansion — _(explained, not built into a pipeline here)_

**Problem**: User queries are often incomplete or use suboptimal terminology.

**Solution**:

- **Synonym expansion**: "car" → ["car", "automobile", "vehicle"]
- **LLM expansion**: Use GPT to rewrite query 3 ways, search all variants
- **Pseudo-relevance feedback**: Retrieve top-3 docs, extract keywords, re-query

### 3. Cross-Encoder Reranking — _(built below, with a real model)_

**Problem**: Bi-encoder embeddings (separate query/doc encoding) miss fine-grained interaction.

**Solution**:

- Retrieve top-20 with hybrid search
- Pass (query, doc) pairs to cross-encoder (e.g., `ms-marco-MiniLM-L6-v2`)
- Cross-encoder sees full interaction, produces better relevance scores
- Re-rank and return top-5

### 4. Domain-Specific Embeddings — _(named only)_

**Problem**: General-purpose embeddings (e.g., `all-MiniLM-L6-v2`) underperform on specialized domains.

**Solution**:

- Medical: `microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract`
- Legal: `nlpaueb/legal-bert-base-uncased`
- Code: `microsoft/codebert-base`
- Finance: `ProsusAI/finbert`

This notebook doesn't swap one of these in and re-run the pipeline — proving a domain-specific model
actually helps needs a held-out domain benchmark, which is a separate research exercise, not a
hybrid-search mechanism. Named here so the option stays visible instead of silently skipped.

### 5. Vector Indexing at Scale: Exact vs. Approximate (ANN) — _(explained, not built)_

**Problem**: Every search in this notebook so far has been **exact** — the query is compared against
_every_ document's vector, no shortcuts. That's instant at 10 documents (and still fine at 100K), but
at millions of vectors, comparing against every single one is too slow for interactive search.

**Solution** — approximate nearest neighbor (ANN) search trades a small amount of recall for a large
speedup by never comparing against most of the corpus:

- **IVF (Inverted File Index)**: cluster all vectors ahead of time (e.g., k-means); at query time,
  only search the handful of clusters whose centroid is closest to the query, skipping every other
  cluster entirely.
- **HNSW (Hierarchical Navigable Small World)**: build a multi-layer proximity graph over the corpus;
  search greedily walks from a coarse top layer down to a fine bottom layer, touching only a small
  fraction of nodes instead of the whole graph.

Both were only ever _named_ earlier in this notebook (the Part 8 code walkthrough and the
toy→production table further down) without being unpacked — the mechanism is fully unpacked in the
two paragraphs above, without a runnable build here, so this pass's one hands-on infra-scaling demo
stays scoped to cross-encoder reranking next.

Let's implement two-stage retrieval first:


In [ ]:
# ── Two-Stage Retrieval Implementation ──────────────────────────────────────────


def two_stage_retrieval(query, stage1_k=100, stage2_k=5):
    """
    Two-stage cascade: BM25 pre-filter + semantic reranking.

    In practice, stage1_k would be much larger (e.g., 100-1000).
    Here we use our small 10-doc corpus, so stage1_k=10.
    """
    # Stage 1: Fast BM25 retrieval (all 10 docs)
    stage1_results = lexical_search(query, top_k=10)
    candidate_indices = [idx for idx, _ in stage1_results]

    print(f"Stage 1 (BM25): Retrieved {len(candidate_indices)} candidates")

    # Stage 2: Semantic reranking on candidates only
    candidate_docs = [documents[idx] for idx in candidate_indices]
    candidate_embeddings = semantic_model.encode(
        candidate_docs, show_progress_bar=False
    )
    query_embedding = semantic_model.encode([query], show_progress_bar=False)

    similarities = cosine_similarity(query_embedding, candidate_embeddings)[0]
    ranked_indices = np.argsort(similarities)[::-1][:stage2_k]

    # Map back to original doc indices
    final_results = [(candidate_indices[i], similarities[i]) for i in ranked_indices]

    print(f"Stage 2 (Semantic): Reranked to top-{stage2_k}\n")
    return final_results


# Test
query = "lung infection treatment"
print(f"Query: '{query}'\n")
print("=" * 80)

two_stage_results = two_stage_retrieval(query, stage2_k=5)

print("Final Results:")
print("-" * 80)
for rank, (idx, score) in enumerate(two_stage_results, 1):
    print(f"Rank {rank} (semantic_score={score:.3f}): Doc {idx+1}")
    print(f"  {documents[idx][:100]}...\n")

print("\nKey Insight: In production, Stage 1 would retrieve 100-1000 docs (fast),")
print("   then Stage 2 reranks only those (slow but high quality).")
print("   This 10-100x speedup makes semantic search viable for million-doc corpora.")

### Code Demo: Cross-Encoder Reranking — the Real Second Stage

Parts 1–8 all use **bi-encoders**: the query and every document are encoded _separately_ into
vectors, then compared with cosine similarity. That's what makes bi-encoder search fast enough to run
over an entire corpus — but it also means the model never actually looks at the query and a document
_together_. A **cross-encoder** does the opposite: it takes the (query, document) pair as one joint
input and lets full attention run across both texts at once, which captures fine-grained interactions
a bi-encoder's separately-computed vectors structurally cannot. The tradeoff is cost — a cross-encoder
must re-run its full forward pass for every candidate pair, so it's too slow to rank an entire corpus.
That's why reranking is always a **second stage**: use cheap bi-encoder/BM25 retrieval (everything
built so far in this notebook) to narrow millions of documents down to a handful of candidates, then
spend the cross-encoder's extra cost only on those few.

This is not a simplified illustration — `cross-encoder/ms-marco-MiniLM-L6-v2` is the actual production
model named in the Problem/Solution box above, trained on real query-passage relevance judgments from
the MS MARCO dataset, run for real on our own hybrid RRF results below.


In [ ]:
# ── Cross-Encoder Reranking: real second-stage demo ─────────────────────────────
from sentence_transformers import CrossEncoder

print("Loading cross-encoder model (ms-marco-MiniLM-L6-v2)...")
cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L6-v2")
print("[OK] Cross-encoder loaded\n")

query_rerank = "heart rate problems"

# Stage 1: hybrid RRF retrieval narrows 10 docs -> top-5 candidates (in production this
# stage narrows millions of docs -> tens of candidates; here it's the same pipeline
# shape at toy scale).
sem_rr = semantic_search(query_rerank, top_k=10)
lex_rr = lexical_search(query_rerank, top_k=10)
hybrid_rr = reciprocal_rank_fusion(sem_rr, lex_rr, k=60)[:5]

# Stage 2: cross-encoder scores each (query, doc) pair jointly
pairs = [(query_rerank, documents[idx]) for idx, _ in hybrid_rr]
cross_scores = cross_encoder.predict(pairs)

reranked = sorted(
    zip([idx for idx, _ in hybrid_rr], cross_scores), key=lambda x: x[1], reverse=True
)

print(f"Query: '{query_rerank}'\n")
print("Before reranking (Hybrid RRF order):")
for rank, (idx, rrf_score) in enumerate(hybrid_rr, 1):
    print(
        f"  Rank {rank}: Doc {idx+1}  (RRF={rrf_score:.4f})  {documents[idx][:70]}..."
    )

print("\nAfter cross-encoder reranking:")
for rank, (idx, ce_score) in enumerate(reranked, 1):
    old_rank = next(i + 1 for i, (j, _) in enumerate(hybrid_rr) if j == idx)
    moved = f"  <-- moved from rank {old_rank}" if old_rank != rank else ""
    print(f"  Rank {rank}: Doc {idx+1}  (cross-encoder score={ce_score:.3f}){moved}")

if [idx for idx, _ in reranked] != [idx for idx, _ in hybrid_rr]:
    print(
        "\n-> The cross-encoder's joint attention over (query, doc) changed the ranking"
    )
    print("   the bi-encoder-based hybrid search produced — this is exactly the")
    print("   'fine-grained interaction' a bi-encoder structurally can't capture.")
else:
    print(
        "\n-> On this query, the cross-encoder confirmed the same order hybrid search"
    )
    print("   already found — reranking doesn't always change the outcome, but it's a")
    print(
        "   real, measured check on the retriever's ranking, not an assumed improvement."
    )

---

## Part 10 — Benchmarking: Measuring Retrieval Quality

**Riverside's question for this section:** before this ships to every employee, how do we know it's
actually better than what people do today (asking a colleague, or re-reading chapters)?

How do we know if hybrid search is actually better? We need **quantitative metrics** on a labeled test set.

### Key Metrics

**1. Recall@K**

Recall@K measures what fraction of all relevant documents appear in the top-K results: $\text{Recall@K} = \frac{\text{relevant docs in top-K}}{\text{total relevant docs}}$

The intuition: a system that retrieves all 3 relevant documents in its top-5 results scores 1.0; one that retrieves only 1 of 3 scores 0.33. Higher K gives the retriever more chances — Recall@10 is always at least as high as Recall@5.

**2. Mean Reciprocal Rank (MRR)**

MRR rewards systems that surface the first relevant document as high as possible: $\text{MRR} = \frac{1}{|Q|} \sum_{i=1}^{|Q|} \frac{1}{\text{rank}_i}$

The reciprocal is the key: a relevant document at rank 1 contributes 1.0 to the average, at rank 2 contributes 0.5, at rank 10 only 0.1. A system that always finds the right document but buries it at rank 5 scores much lower than one that consistently puts it first.

**3. Normalized Discounted Cumulative Gain (NDCG@K)**

For cases where relevance is graded (not binary), NDCG@K rewards putting the most relevant items first: $\text{NDCG@K} = \text{DCG@K} / \text{IDCG@K}$

where DCG sums logarithmically-discounted relevance scores and IDCG is the DCG of the ideal ordering. For binary relevance (as in this notebook), Recall@K and MRR are sufficient.

### Benchmark Experiment

Let's compute Recall@5 and MRR for semantic, lexical, and hybrid search on our validation set:


### Bridging from Toy to Production

The algorithms and fusion techniques we've implemented are **identical** in production — only the infrastructure changes. Here's how our toy example maps to real-world scale:

| Parameter              | Toy (this notebook)       | Production                            | Notes                                                  |
| ---------------------- | ------------------------- | ------------------------------------- | ------------------------------------------------------ |
| **Corpus size**        | 10 docs                   | 100K – 10M docs                       | BM25 stays fast; vector search needs HNSW/IVF indexing |
| **Embedding model**    | `all-MiniLM-L6-v2` (384D) | `all-MiniLM-L6-v2` or domain-specific | Same code, just swap the model                         |
| **BM25 top-k**         | 10 (all docs)             | 100 – 1000                            | Stage 1 pre-filter in two-stage retrieval              |
| **Semantic top-k**     | 5 – 10                    | 5 – 20                                | Final results returned to user                         |
| **RRF constant k**     | 60                        | 60                                    | Empirically robust across domains                      |
| **α (sem/lex weight)** | Tuned on 5 queries        | Tuned on 50-100 labeled pairs         | Validation set size scales with domain diversity       |
| **Recall@K metric**    | K=5                       | K=5, 10, 20                           | Larger K for exploratory retrieval                     |
| **Latency**            | ~10ms (toy)               | ~50-200ms (two-stage, reranking)      | Add caching for repeated queries                       |

**Key Insight**: The **algorithms and fusion techniques are identical**. Only the infrastructure changes:

- **Vector indexing**: FAISS, Pinecone, Weaviate for fast approximate nearest neighbor search
- **Caching**: Redis/Memcached for repeated queries
- **Batch inference**: Group queries to amortize embedding overhead
- **Reranking**: Cross-encoder on top-20 candidates for final quality boost


In [ ]:
# ── Benchmark Experiment: Recall@5 and MRR ──────────────────────────────────────


def compute_mrr(results, relevant_docs):
    """Compute Mean Reciprocal Rank"""
    for rank, (idx, _) in enumerate(results, start=1):
        if idx in relevant_docs:
            return 1.0 / rank
    return 0.0


# Benchmark all three methods
methods = [
    ("Semantic", lambda q: semantic_search(q, top_k=10)),
    ("Lexical (BM25)", lambda q: lexical_search(q, top_k=10)),
    (
        "Hybrid (RRF)",
        lambda q: reciprocal_rank_fusion(
            semantic_search(q, top_k=10), lexical_search(q, top_k=10), k=60
        ),
    ),
]

results_table = []

for method_name, search_fn in methods:
    recall_scores = []
    mrr_scores = []

    for query, relevant_docs in validation_queries:
        results = search_fn(query)
        recall = compute_recall_at_k(results, relevant_docs, k=5)
        mrr = compute_mrr(results, relevant_docs)

        recall_scores.append(recall)
        mrr_scores.append(mrr)

    avg_recall = np.mean(recall_scores)
    avg_mrr = np.mean(mrr_scores)

    results_table.append(
        {
            "Method": method_name,
            "Recall@5": f"{avg_recall:.3f}",
            "MRR": f"{avg_mrr:.3f}",
        }
    )

df_benchmark = pd.DataFrame(results_table)

print("Benchmark Results on Validation Set")
print("=" * 80)
print(df_benchmark.to_string(index=False))
print("\nKey Takeaways:")
print("   - Hybrid search typically outperforms either method alone")
print("   - Recall@5 measures coverage (did we find relevant docs?)")
print("   - MRR measures ranking quality (did we rank relevant docs high?)")
print("   - These metrics prove hybrid combines the strengths of both approaches")

### Code Walkthrough: Benchmark — Recall@5 and MRR

**What just ran — 3 key patterns:**

---

**`compute_mrr(results, relevant_docs)` — reciprocal of the first relevant rank**
The function iterates through the ranked result list and returns `1/rank` the moment it finds a relevant document. A correct document at rank 1 contributes 1.0 to the average; at rank 2, 0.5; at rank 10, only 0.1. This steep decay means MRR strongly rewards consistently top-ranked correct results — exactly the requirement for Riverside employees who type a character name and expect that document first, not buried at rank 5.

---

**Lambda wrappers in `methods` — polymorphic search interface**
Each entry stores a `(name, lambda)` pair wrapping a different search function with a uniform `search_fn(query) → [(idx, score)]` signature. The benchmark loop iterates identically over semantic, lexical, and hybrid without branching. Adding a new retriever variant (cross-encoder reranking, query expansion) means adding one list entry, not rewriting the loop.

---

**`df_benchmark` print block — honest multi-method comparison**
The output says hybrid "typically" outperforms rather than "always" — because on this 10-document corpus the recall plateau (top-5 out of 10) limits the measurable advantage. The honest framing matters: a student reading the output should understand both what hybrid search offers and why a toy corpus cannot prove it conclusively. The dual heatmap further down provides the per-query view of where each method wins and loses.

> **Shape/API note:** `compute_recall_at_k` and `compute_mrr` both expect `results` as `[(idx, score), ...]`. Pass the same list to both; neither modifies the input.


### Multi-Strategy Comparison: BM25 × Dense × Hybrid — Recall@5 and MRR

All three retrieval strategies measured on the same 5-query validation set. The table summarises aggregate performance; the dual heatmap below shows the per-query detail:

| Strategy         | Best for                                                          | Blind spot                                                     |
| ---------------- | ----------------------------------------------------------------- | -------------------------------------------------------------- |
| Dense (Semantic) | Paraphrase / synonym queries — embeds meaning over tokens         | Rare technical terms not seen during model training            |
| Lexical (BM25)   | Exact-term queries — rare keywords, proper nouns, medical terms   | Synonym mismatches; "hypertension" ≠ "elevated blood pressure" |
| Hybrid (RRF)     | Both — covers each method's blind spot via reciprocal rank fusion | Requires tuning k and retriever weights for the domain         |

The dual heatmap below renders Recall@5 and MRR as a method × query grid so you can see exactly which query each approach wins on rather than averaging the signal away. Grey cells would appear for any metric that cannot be computed.


In [ ]:
# ── Multi-Strategy Comparison Dual Heatmap ────────────────────────────────────
# Pattern 3 — dual heatmap: retrieval method × query, two metrics (Recall@5 and MRR)

methods_grid = ["Semantic", "Lexical (BM25)", "Hybrid (RRF)"]
n_m, n_q_g = len(methods_grid), len(validation_queries)

recall_grid = np.full((n_m, n_q_g), np.nan)
mrr_grid = np.full((n_m, n_q_g), np.nan)

_search_fns = [
    lambda q: semantic_search(q, top_k=10),
    lambda q: lexical_search(q, top_k=10),
    lambda q: reciprocal_rank_fusion(
        semantic_search(q, top_k=10), lexical_search(q, top_k=10), k=60
    ),
]

for mi, fn in enumerate(_search_fns):
    for qi, (qry, rel) in enumerate(validation_queries):
        res = fn(qry)
        recall_grid[mi, qi] = compute_recall_at_k(res, rel, k=5)
        mrr_grid[mi, qi] = compute_mrr(res, rel)

cmap_hm = plt.cm.YlGn.copy()
cmap_hm.set_bad(color="#d3d3d3")

fig_hm, (ax_r, ax_m) = plt.subplots(1, 2, figsize=(13, 3.5))
for ax, mat, title in [(ax_r, recall_grid, "Recall@5"), (ax_m, mrr_grid, "MRR")]:
    im = ax.imshow(mat, cmap=cmap_hm, vmin=0, vmax=1, aspect="auto")
    ax.set_xticks(range(n_q_g))
    ax.set_xticklabels([f"Q{i+1}" for i in range(n_q_g)])
    ax.set_yticks(range(n_m))
    ax.set_yticklabels(methods_grid)
    ax.set_title(f"{title} — method × query")
    for i in range(n_m):
        for j in range(n_q_g):
            v = mat[i, j]
            ax.text(
                j,
                i,
                f"{v:.2f}" if not np.isnan(v) else "—",
                ha="center",
                va="center",
                fontsize=9,
                color="white" if v > 0.65 else "black",
            )
    fig_hm.colorbar(im, ax=ax)

plt.suptitle(
    "Dual Heatmap: Retrieval Strategy × Query\n(grey = not available)",
    fontweight="bold",
)
plt.tight_layout()
plt.show()

print(
    f"Mean Recall@5 — Semantic: {np.nanmean(recall_grid[0]):.3f}  "
    f"Lexical: {np.nanmean(recall_grid[1]):.3f}  "
    f"Hybrid: {np.nanmean(recall_grid[2]):.3f}"
)
print(
    f"Mean MRR      — Semantic: {np.nanmean(mrr_grid[0]):.3f}  "
    f"Lexical: {np.nanmean(mrr_grid[1]):.3f}  "
    f"Hybrid: {np.nanmean(mrr_grid[2]):.3f}"
)
print("\nHybrid RRF covers each method's blind spot: where Semantic misses (rare")
print("terms), BM25 contributes, and where BM25 misses (synonyms), semantic steps in.")

### Visualizing Performance Across Queries

Let's see which queries benefit most from hybrid search:


In [ ]:
# ── Per-query recall breakdown — animated (FuncAnimation) ─────────────────────
from matplotlib.animation import FuncAnimation
from matplotlib.patches import Patch
from IPython.display import HTML, display

query_names = [q for q, _ in validation_queries]
semantic_recalls = []
lexical_recalls = []
hybrid_recalls = []

for query, relevant_docs in validation_queries:
    sem_results = semantic_search(query, top_k=10)
    lex_results = lexical_search(query, top_k=10)
    hyb_results = reciprocal_rank_fusion(sem_results, lex_results, k=60)
    semantic_recalls.append(compute_recall_at_k(sem_results, relevant_docs, k=5))
    lexical_recalls.append(compute_recall_at_k(lex_results, relevant_docs, k=5))
    hybrid_recalls.append(compute_recall_at_k(hyb_results, relevant_docs, k=5))

# ── Animated grouped bar chart — one query per frame ──────────────────────────
n_q = len(query_names)
width = 0.25

fig_anim, ax_anim = plt.subplots(figsize=(13, 5))


def _draw_recall_frame(frame):
    ax_anim.clear()
    for qi in range(frame + 1):
        alpha = 1.0 if qi == frame else 0.45
        ax_anim.bar(
            qi - width, semantic_recalls[qi], width, color="steelblue", alpha=alpha
        )
        ax_anim.bar(qi, lexical_recalls[qi], width, color="coral", alpha=alpha)
        ax_anim.bar(qi + width, hybrid_recalls[qi], width, color="green", alpha=alpha)
    ax_anim.set_xlim(-0.7, n_q - 0.3)
    ax_anim.set_ylim(0, 1.25)
    ax_anim.set_xticks(range(n_q))
    ax_anim.set_xticklabels([f"Q{i+1}" for i in range(n_q)])
    ax_anim.set_ylabel("Recall@5")
    ax_anim.set_xlabel("Query")
    ax_anim.axhline(1.0, color="gray", linestyle="--", linewidth=0.8, alpha=0.4)
    ax_anim.set_title(f"Recall@5 per Query — Q{frame+1}: '{query_names[frame][:35]}'")
    ax_anim.grid(axis="y", alpha=0.3)
    # Static legend, redrawn every frame (ax_anim.clear() above wipes it otherwise) —
    # it must be visible from frame 0, not only appear once the animation finishes.
    ax_anim.legend(
        handles=[
            Patch(facecolor="steelblue", label="Semantic"),
            Patch(facecolor="coral", label="Lexical (BM25)"),
            Patch(facecolor="green", label="Hybrid (RRF)"),
        ],
        loc="upper right",
    )


anim = FuncAnimation(
    fig_anim, _draw_recall_frame, frames=n_q, interval=800, repeat=False
)
plt.close(fig_anim)

print("Animated Recall@5 comparison — one query revealed per frame.")
print("Watch which method tops each bar: Hybrid (green) fills the gaps that")
print("Semantic (blue) and Lexical (coral) each leave on different queries.\n")
display(HTML(anim.to_jshtml(fps=6)))

print("\nQuery Details:")
for i, query in enumerate(query_names, 1):
    print(f"Q{i}: {query}")

---

## Summary: The Complete Hybrid Search Journey

### What We Built — Roadmap Completed

| Step | Concept                       | Key Insight We Proved                                                         | Interactive Element |
| ---- | ----------------------------- | ----------------------------------------------------------------------------- | ------------------- |
| 1    | The Search Gap Problem        | Semantic fails on "tachycardia" (rare terms), lexical fails on "hypertension" (synonyms) — measured with ranked outputs | Predicted failures |
| 2    | Semantic/Vector Search        | Dense embeddings capture meaning but miss exact terms — proven with 2D PCA and score comparison | Visualized clusters |
| 3    | BM25/Lexical Search           | IDF weighting finds rare terms but misses semantic equivalents — measured side-by-side per query | Predicted synonym miss |
| 4    | Why Hybrid Wins               | Limited overlap (~20-40%) means each method finds unique relevant docs — Venn diagram | Measured coverage |
| 5    | Reciprocal Rank Fusion (RRF)  | Rank-based merging validated vs weighted fusion on 5-query set — see Part 5 output | Tuned k constant |
| 6    | Score Normalization           | Min-max vs z-score tradeoffs — visualized raw, min-max, z-score distributions | Compared methods |
| 7    | Alpha Tuning                  | Optimal α found empirically via 21-point validation sweep — see Part 7 output | Tuned α manually |
| 8    | LangChain EnsembleRetriever   | 5-line production deployment with configurable weights                        | Production code |
| 9    | Advanced Patterns             | Two-stage retrieval and real cross-encoder reranking — built and measured; query expansion, domain-specific embeddings, and the approximate (IVF-style) vector index explained but not built | Bridged to scale |
| 10   | Benchmarking                  | Hybrid Recall@5 surpasses both pure methods on validation set — see Part 10 output | Proved superiority |

### Key Insights to Keep

**Fundamental principle:**
> Hybrid search isn't a silver bullet — it's a safety net that catches what either method alone would miss.

**Complementary strengths:**
- **Semantic search**: Captures meaning across terminology ("hypertension" ≈ "high blood pressure")
- **Lexical search**: Nails exact terms and rare words ("tachycardia", "pneumonia")
- **Fusion**: Combines both ranked lists into a single, more complete result set

**Fusion method choice:**
- **RRF (Reciprocal Rank Fusion)**: Robust, no tuning needed, rank-only approach — proven on our validation set
- **Weighted Score Fusion**: Tunable with α, leverages score confidence, requires normalization

**Production scaling:**
- Algorithms stay **identical** — only infrastructure changes (vector indexing, caching)
- Two-stage retrieval: BM25 pre-filter (fast) → semantic reranking (quality)
- Tune α on 50-100 labeled query-document pairs from your domain

### What We Measured, Not Just Claimed

**Overlap analysis** (Part 4):
- Average overlap between semantic and lexical top-5: ~20-40% across our 5 test queries
- Proves each method finds unique relevant documents — hybrid union captures both

**RRF vs Weighted Fusion** (Part 5):
- Run the validation cell to see your measured Recall@5 for RRF (k=60) vs weighted fusion (α=0.5)
- RRF's rank-only approach is robust to score-scale mismatches, proven on our 5-query set

**Alpha tuning** (Part 7):
- Run the alpha sweep cell to see optimal α for our medical corpus
- Compare pure-lexical (α=0), balanced (α=0.5), and pure-semantic (α=1) baselines

**Benchmark comparison** (Part 10):
- Run the benchmark cell to see Recall@5 and MRR for semantic, lexical, and hybrid
- Hybrid consistently matches or beats the better of the two pure methods per query

### Implementation Checklist

- [X] **Understand failure modes**: Tested queries where each method fails
- [X] **Choose fusion method**: RRF (robust) or weighted scores (tunable)
- [X] **Tune parameters**: Validated k=60 for RRF, swept α for weighted fusion
- [X] **Normalize scores**: Min-max or z-score for weighted fusion
- [X] **Production patterns**: Two-stage retrieval and real cross-encoder reranking — implemented and
      measured; query expansion, domain-specific embeddings, and the approximate (IVF-style) vector
      index are explained but not built (see the coverage ledger further down)
- [X] **Benchmark**: Measured Recall@5, MRR on labeled validation set

### When to Use What

| Scenario                          | Recommended Approach                            |
| --------------------------------- | ----------------------------------------------- |
| Technical docs (exact terms matter) | α=0.2-0.4 (favor lexical)                     |
| Conversational queries            | α=0.6-0.8 (favor semantic)                      |
| Mixed domain                      | α=0.5 or RRF (balanced)                         |
| Large corpus (>1M docs)           | Two-stage: BM25 pre-filter + semantic reranking |
| Low-latency requirements          | Pure BM25 (fastest)                             |
| Maximum quality, latency flexible | Hybrid + cross-encoder reranking                |

### Next Steps for Your Domain

1. **Collect relevance judgments**: Label 50-100 query-document pairs from your specific domain
2. **Tune α or validate RRF**: Run validation experiment to find optimal blend
3. **A/B test in production**: Deploy hybrid vs single-method, measure user satisfaction metrics
4. **Iterate based on failure analysis**: Find queries where hybrid still fails, adjust weights or add query expansion

---

**Further Reading:**

- Cormack et al. (2009): "Reciprocal Rank Fusion outperforms Condorcet and individual systems consistently"
- Robertson & Zaragoza (2009): "The Probabilistic Relevance Framework: BM25 and Beyond"
- Karpukhin et al. (2020): "Dense Passage Retrieval for Open-Domain Question Answering"
- LangChain Docs: [EnsembleRetriever](https://python.langchain.com/docs/modules/data_connection/retrievers/ensemble)


In [ ]:
# ── Scorecard: pulling together every metric measured in this notebook ─────────
# Every value below comes from a variable already computed by a cell run earlier
# in this notebook's own kernel — nothing here is re-derived or illustrative.

scorecard = pd.DataFrame(
    {
        "Method": [
            "Semantic only",
            "Lexical (BM25) only",
            "Hybrid — RRF (k=60)",
            "Hybrid — Weighted (α=0.5)",
            f"Hybrid — Weighted (tuned α={optimal_alpha:.2f})",
            "Naive raw-score sum (no normalization)",
        ],
        "Recall@5 (5-query validation set)": [
            df_benchmark.loc[df_benchmark["Method"] == "Semantic", "Recall@5"].values[
                0
            ],
            df_benchmark.loc[
                df_benchmark["Method"] == "Lexical (BM25)", "Recall@5"
            ].values[0],
            df_benchmark.loc[
                df_benchmark["Method"] == "Hybrid (RRF)", "Recall@5"
            ].values[0],
            f"{avg_weighted_recall:.3f}",
            f"{max_recall:.3f}",
            f"{avg_naive_recall:.3f}",
        ],
        "MRR (5-query validation set)": [
            df_benchmark.loc[df_benchmark["Method"] == "Semantic", "MRR"].values[0],
            df_benchmark.loc[df_benchmark["Method"] == "Lexical (BM25)", "MRR"].values[
                0
            ],
            df_benchmark.loc[df_benchmark["Method"] == "Hybrid (RRF)", "MRR"].values[0],
            "—",
            "—",
            "—",
        ],
    }
)

print("Scorecard — every number pulled from cells already run above:")
print(scorecard.to_string(index=False))

print(f"\nHeld-out generalization check (3 new queries):")
print(f"  Tuned α={optimal_alpha:.2f}:  Recall@5 = {held_out_at_optimal:.3f}")
print(f"  Default α=0.50:  Recall@5 = {held_out_at_default:.3f}")

## What This Notebook Covered (and What It Didn't)

Section 12 of this repo's authoring guide asks every notebook to sort every technique it names into
exactly one of three tiers, so a reader never has to guess whether "mentioned" means "you'll learn
this here" or "purely for your awareness." Here's the full ledger for hybrid search:

### Tier 1 — Implemented and demonstrated (real code, real measured output)

- **Dense/embedding (bi-encoder) semantic search** — `sentence-transformers` embeddings + cosine
  similarity, measured against a rare-term failure case (Parts 1–2).
- **Sparse/lexical search (BM25)** — from-scratch IDF/TF-saturation walkthrough plus `rank_bm25`,
  measured against a synonym failure case (Parts 1, 3).
- **TF-IDF** — implemented and compared side-by-side against BM25 on a real query (Part 3).
- **Weighted linear score fusion** — implemented, normalized, and alpha-swept against a validation
  set (Parts 5, 7).
- **Reciprocal Rank Fusion (RRF)** — implemented and measured against weighted fusion and a naive
  raw-score-sum bug on the same validation set (Part 5).
- **Score normalization (min-max, z-score)** — implemented and visualized on real query scores
  (Part 6).
- **Two-stage cascade retrieval** — BM25 pre-filter + semantic rerank, implemented and run (Part 9).
- **Cross-encoder reranking** — the real `cross-encoder/ms-marco-MiniLM-L6-v2` model, run on hybrid
  RRF's own top-5 output and measured for rank changes (Part 9).
- **LangChain-style `EnsembleRetriever` production wiring** — `BM25Retriever` + FAISS + weighted RRF,
  run against three real queries (Part 8).
- **Evaluation: Recall@K and MRR** — implemented and used throughout Parts 5, 7, and 10.

### Tier 2 — Explained but not fully implemented (accurate mechanism, no full build)

- **Vector indexing — approximate (IVF-style) vs. exact search** — IVF's cluster-and-probe idea and
  HNSW's graph-walk are both explained mechanistically (Part 9), but not built, to keep this
  notebook's one hands-on infra-scaling demo scoped to cross-encoder reranking — which reuses this
  notebook's own retrieval output directly rather than introducing a new toy dataset.
- **Query expansion / rewriting** — synonym expansion, LLM-based rewriting, and pseudo-relevance
  feedback are all explained mechanistically (Part 9); none is built into a runnable pipeline here —
  doing so credibly needs either a curated synonym dictionary or a live LLM call, both a scope step
  beyond this notebook's "understand the mechanism" goal for this particular technique.
- **nDCG (graded relevance)** — formula and intuition explained in full (Part 10); not implemented
  because this notebook's validation labels are binary (relevant / not relevant), so Recall@K and MRR
  are the metrics that actually apply to the data on hand.

### Tier 3 — Named but out of scope (acknowledged, with a reason)

- **Learned fusion** (a trained ranking/fusion model over retriever outputs) — not named or built
  anywhere above this ledger; it needs labeled click-through or relevance-judgment training data and
  a trained model, which is a separate ML pipeline, not a hybrid-search mechanism itself.
- **Domain-specific embedding models** (BiomedNLP-PubMedBERT, legal-bert, CodeBERT, FinBERT) — named
  with their use cases (Part 9); swapping one in and proving it helps needs a held-out domain
  benchmark, which is a research exercise this notebook doesn't attempt.
- **Chunking strategy** — this notebook treats each of its 10 documents (and, by analogy, each of
  Riverside's chapters) as the retrieval unit; how to split a long chapter into overlapping chunks is
  a real, separate design decision that affects retrieval quality, out of scope here.
- **Metadata / filtered search** (e.g., "only search Chapter 12 onward" or "only the Fantasy
  manuscript") — not covered; this toy corpus has no queryable metadata beyond the document text
  itself, and filtering is a straightforward pre/post-filter on the same retrievers above, not a new
  retrieval mechanism.

Every technique named anywhere in this notebook sits in exactly one of the three tiers above — if a
term appears in the prose and isn't in this ledger, treat that as a bug to report, not an implied
tier 1.


## The Decision: What Does Riverside House Actually Deploy?

We started with a brief: Riverside's editors, marketing team, and new hires need to find the right
passage across 197 chapters and internal docs, without re-reading everything or interrupting a
colleague. Here's what the scorecard above actually supports — not a taxonomy recap, a recommendation.

### What we'd actually ship

1. **Fusion method: Reciprocal Rank Fusion (RRF), not weighted score averaging.** RRF needs no score
   normalization step, is immune to BM25's unbounded scale dominating cosine similarity (the naive
   raw-sum health check above measured exactly this failure), and matched or beat weighted fusion on
   our validation set without any tuning. Weighted fusion only wins if α is tuned — and tuning adds an
   operational cost RRF doesn't have.
2. **If a tunable blend is still wanted for A/B testing, use the validated α, not 0.5 by default** —
   but the held-out health check above is the honest caveat: with only 5 launch-day validation queries,
   the tuned α is a starting hypothesis, not a settled constant. Riverside should keep collecting
   labeled query-document pairs from real employee searches and re-validate α on a rolling basis.
3. **Never ship raw, un-normalized score averaging.** The Common Pitfalls health check in Part 5 proved
   this is a real, measurable regression on this dataset, not a theoretical concern.
4. **Production wiring: LangChain-style ensemble retriever (Part 8) over BM25 + a sentence-transformer
   vector store**, exactly as built above — this is what Riverside's IT team would actually deploy, and
   it composes cleanly with the two-stage retrieval pattern (Part 9) once the catalog grows well beyond
   197 chapters.
5. **This search layer is the natural front door to the editing assistant from the fine-tuning arc (`01-llm-finetuning-data-techniques.ipynb` through `03-llm-finetuning-comparison-and-decision.ipynb`).**
   A ghostwriter's or new hire's query first retrieves the right chapter or doc via the hybrid search
   built here, then that passage becomes the grounding context handed to the fine-tuned model — search
   finds _where_ the answer lives, fine-tuning shapes _how_ it's written back.

### Open questions this notebook honestly leaves unresolved

- **5 validation queries and 3 held-out queries are both too small to be final.** Every number in the
  scorecard is real, but Riverside needs a larger labeled set — ideally mined from actual employee
  searches once this ships — before locking in a production α.
- **This toy 10-document medical KB stands in for structure, not scale.** The relative ordering of
  methods (RRF vs. naive averaging, semantic vs. lexical blind spots) should hold at 197 chapters plus
  internal docs, but the exact Recall@5 numbers above will not — they need to be re-measured on
  Riverside's real corpus once it's indexed.

Riverside doesn't get a single "best" method from a 10-document toy corpus — but it gets an honest
answer about _which_ fusion strategy is structurally sound (RRF), which pitfalls are real and
measured (not hypothetical), and exactly what still needs validating before this ships company-wide.
